# Reproducing Gao et al. (2018), "A cortico-cerebellar loop for motor planning", from dandiset 000007

**Paper.** Gao, Davis, Thomas, Economo, Abrego, Svoboda, De Zeeuw & Li (2018), Nature 563, 113-116,
doi:10.1038/s41586-018-0633-x. The claims tested are those of `claims/gao-2018.yaml`, written by a
reader from the NIH author manuscript (PMC6212318) with its Methods, figure legends, Supplementary
Information and reporting summary. The paper links no code.

**Dataset.** DANDI:000007, "A cortico-cerebellar loop for motor planning", version **0.220126.1903**
(doi:10.48324/dandi.000007/0.220126.1903). The paper cites no version of the dataset. This is the only
published version, and the draft holds the same 54 assets (same count and total size, 199,439,472
bytes), so the most recent published version is pinned. Files are streamed from the archive; nothing
is written to disk.

**What the dataset holds.** 54 ALM recording sessions in 13 mice, the 13 ALM-recorded mice of
Supplemental Table 1: 4 L7-cre x Ai32 mice with CN photo-inhibition, 6 wild-type mice with fastigial
and 3 with dentate ChR2 photo-activation. Each file has a trials table (event times, outcome, an
undocumented photostimulation code) and sorted units with mean waveforms. There are no lick times, no
CN recordings, no behavior-only sessions (lesions, sub-epoch photo-inhibition, response epoch
photostimulation, VGAT-ChR2 experiments) and no anatomy.

| claim | tested here |
|---|---|
| fig1d-fastigial-lesion-contra-choice, fig1e-fastigial-lesion-slow-initiation, fig1d-dentate-lesion-no-effect | blocked, data_missing (lesion mice not in the dataset) |
| fig1g-fastigial-activation-early-sample-no-effect | tested, recording sessions (code 1) |
| fig1g-fastigial-activation-delay-errors | tested, recording sessions (code 2) |
| fig1h-fastigial-activation-delay-bias | tested, reading (a) only (early delay) |
| fig1h-fastigial-activation-response-contra-bias | blocked, data_missing (no response epoch stimulation) |
| fig1i-fastigial-activation-licking-unaffected | blocked, data_missing (no lick times); the early lick part is shown |
| fig1g-dentate-activation-little-effect | tested, reading (a), 3 of 8 mice |
| fig1j-photoinhibition-silences-fastigial | blocked, data_missing (CN recordings) |
| fig1k-inhibition-early-sample-no-effect, fig1k-inhibition-late-sample-delay-errors | blocked, data_missing (no 0.5 s photo-inhibition) |
| fig1k-inhibition-delay-bias | tested, reading (b) (1.3 s delay photo-inhibition), recording sessions |
| fig1k-inhibition-response-ipsi-bias, fig1l-inhibition-execution-unaffected | blocked, data_missing |
| ed10-vgat-fastigial-perturbation-performance, ed10ef-alm-drives-licking-after-fastigial, ed10c-alm-inhibition-endpoints-predict | blocked, data_missing (VGAT-ChR2 mice not in the dataset) |
| fig2-alm-selective-fraction | tested |
| fig2-cn-selective-fraction, fig2-preparatory-more-in-alm, fig2-peri-movement-equal | blocked, data_missing (CN recordings); ALM fractions shown |
| ed4-lick-modulated-few | blocked, data_missing (no lick times) |
| fig2cd-selectivity-emerges-gradually | tested on ALM; the CN half is not in the dataset |
| ed4e, ed4f, ed4g, fig3-cn-altered, fig3b, fig3c, ed6 (CN claims) | blocked, data_missing |
| fig3d-alm-altered-by-cn-inhibition, fig3e-alm-mean-rate-unchanged, fig3e-alm-increase-decrease, fig3f-alm-selectivity-abolished-by-cn-inhibition | tested |
| fig4bc-fastigial-targets-alm-thalamus, ed8-snr-sc-converge | blocked, data_missing (anatomy) |
| fig4-fastigial-activation-alm-latency, fig4e-fastigial-activation-abolishes-alm-selectivity, fig4f-fastigial-effect-persists, fig4g-dentate-effect-recovers | tested |
| fig4i-cd-control-separation, fig4i-fastigial-collapses-cd, cd-captures-selectivity, ed10d-fastigial-endpoints-no-prediction | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import resource
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import gao2018 as g

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
N_BOOT = 10_000   # B1 and E5
N_CI = 1000       # bootstrap intervals over neurons or sessions (a standard choice)

In [ ]:
S = g.load_all()
P = g.prepare(S)
print(f"loaded {len(S)} sessions in {time.time() - T0:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

What was loaded, by subject, session and photostimulation code; the units, every column of the units
table and every group of the files; the event times; what the undocumented `photo_stim_id` codes are;
and whether the spikes share the clock of the trials table.

In [ ]:
rows = []
for s, p in zip(S, P):
    t = g.add_trial_fields(s.trials)
    w = g.spike_width_ms(s.waveforms, s.wf_rate)
    rows.append(dict(subject=s.subject, session=s.path.split("_ses-")[1][:15], genotype=s.genotype,
                     site=s.site, site_hemisphere=s.site_hemisphere, rec_ml=s.rec_ml, group=p.group,
                     trials=len(t), lick_early=int(t.lick_early.sum()), units=s.n_units,
                     pyr_width=int((w > g.PYR_MIN_MS).sum()), fs_width=int((w < g.FS_MAX_MS).sum()),
                     pyr_file=int((s.cell_type == "pyramidal").sum()),
                     codes=",".join(f"{k}:{v}" for k, v in t.stim.value_counts().sort_index().items())))
D = pd.DataFrame(rows)
print(D.to_string())
by = D.groupby(["group", "subject"]).agg(sessions=("session", "size"), trials=("trials", "sum"),
                                         units=("units", "sum"), pyr_width=("pyr_width", "sum"),
                                         pyr_file=("pyr_file", "sum"), site=("site", "first"),
                                         site_hemisphere=("site_hemisphere", "first"), rec_ml=("rec_ml", "first"))
print(by.to_string())
print(D.groupby("group")[["units", "pyr_width", "fs_width", "pyr_file"]].sum().assign(
    sessions=D.groupby("group").size(), mice=D.groupby("group").subject.nunique()))
print("total units", D.units.sum(), "; paper: 1309 units in 10 mice and 39 sessions with CN photo-activation "
      "(912 putative pyramidal in the 9 analysed mice, 446 fastigial and 466 dentate), 490 units (454 pyramidal) "
      "in 4 mice and 19 sessions with CN photo-inhibition")

In [ ]:
# Units: widths from the mean waveforms against the files' cell_type column; sampling rate.
W = []
for s in S:
    w = g.spike_width_ms(s.waveforms, s.wf_rate)
    W.append(pd.DataFrame(dict(subject=s.subject, width=w, cell_type=s.cell_type, wf_len=s.waveforms.shape[1],
                               rate=s.wf_rate)))
W = pd.concat(W)
W["by_width"] = np.where(W.width > g.PYR_MIN_MS, "pyramidal (>0.45 ms)",
                         np.where(W.width < g.FS_MAX_MS, "FS (<0.35 ms)", "intermediate"))
print(pd.crosstab(W.cell_type, W.by_width, margins=True))
print("waveform sampling rate (Hz):", W.rate.unique(), "; samples per waveform:", sorted(W.wf_len.unique()),
      "; width range", W.width.min(), "to", W.width.max(), "ms, in steps of", 1000 / W.rate.iloc[0], "ms")
# The Methods give 25 kHz for the 32-channel NeuroNexus recordings ("demultiplexed (sampling at 25,000 Hz)");
# the stated waveform sampling rate per probe type, from the electrode group:
print(pd.DataFrame([dict(probe=s.probe, rate_Hz=s.wf_rate) for s in S]).value_counts().rename("sessions").to_string())
# spike times: sorted, within the session, and units with few spikes
nsp = np.array([len(x) for s in S for x in s.spikes])
srt = all(np.all(np.diff(x) >= 0) for s in S for x in s.spikes)
print("spikes per unit: min", nsp.min(), "median", int(np.median(nsp)), "max", nsp.max(), "; all sorted:", srt)

In [ ]:
# Every column of the units table and every top-level group of the files. There is no quality or
# isolation field to explain the unit counts, and no acquisition or processing data (no lick or video
# channel).
print("units table columns:", sorted({tuple(s.units_columns) for s in S}))
for k in sorted({k for s in S for k in s.groups}):
    if k != "specifications":
        print(f"  group {k!r}: members {sorted({m for s in S for m in s.groups.get(k, [])})}")
print("trials (no lick early) outside the units' obs_intervals, left out of the neural analyses:",
      {p.path.split("/")[1][:-4]: p.n_unobserved for p in P if p.n_unobserved})
for p in P:
    if p.n_unobserved:
        s_ = S[p.idx]
        ends = [iv[:, 1].max() for iv in s_.obs_intervals]
        out = ~g.observed(s_)
        print(f"  {p.path.split('/')[1][:-4]}: obs_intervals end at {min(ends):.1f} to {max(ends):.1f} s; trials "
              f"outside them (all trials) {int(out.sum())} of {len(s_.trials)}, the last {int(out[::-1].cumprod().sum())} "
              f"in a row, from {s_.trials.start_time[out].min():.1f} to {s_.trials.stop_time.max():.1f} s; "
              f"spikes after the end: {sum(int((x > max(ends)).sum()) for x in s_.spikes)}")
# The files hold more putative pyramidal neurons than the paper in every group, and the paper names no
# selection beyond spike width. Delay-epoch rate on control trials, per group: how many neurons are slow,
# and the mean rate of the n fastest where n is the paper's count. Shown to bound what an unstated
# selection by rate could do; it is not used by any claim.
UR = pd.DataFrame([dict(group=p.group, subject=p.subject, rate=p.ep["delay"][j][(p.t.stim == "0").to_numpy()].mean() / 1.3)
                   for p in P for j in range(len(p.units))])
for gr, n_paper in (("fastigial", 446), ("dentate", 466), ("inhibition", 454)):
    x = np.sort(UR[UR.group == gr].rate.to_numpy())[::-1]
    print(f"{gr}: {len(x)} neurons (paper {n_paper}); below 0.5 spikes/s {int((x < 0.5).sum())}, below 1 {int((x < 1).sum())}; "
          f"mean rate all {x.mean():.2f}, of the {n_paper} fastest {x[:n_paper].mean():.2f} (slowest kept {x[n_paper - 1]:.2f})")
print("putative pyramidal neurons per session and group: see the first table (pyr_width); the paper gives no per-session counts")


In [ ]:
# Event times. The column descriptions say pole_in_time, pole_out_time and cue_time are relative to the
# trial start; the values are on the session clock (start_time + 0.57 s for pole_in_time).
T = pd.concat([g.add_trial_fields(s.trials).assign(subject=s.subject, session=i, group=p.group)
               for i, (s, p) in enumerate(zip(S, P))])
print(S[0].trials.columns.tolist())
print("pole_in_time - start_time (s):", (T.pole_in_time - T.start_time).describe().round(3).to_dict())
print("sample epoch (pole_out - pole_in) and delay epoch (cue - pole_out), trials without lick early:")
print(T[T.lick_early == 0][["sample_s", "delay_s"]].describe().round(4))
print("with lick early:", T[T.lick_early == 1][["sample_s", "delay_s"]].describe().round(3).loc[["count", "min", "50%", "max"]])
print("trials without lick early whose epochs are not 1.3 s:", int((~T.standard & (T.lick_early == 0)).sum()))
print("trial duration stop - start (s):", (T.stop_time - T.start_time).unique()[:5])
print(pd.crosstab(T.group, T.response))
print("duplicate recordings: identical trial tables",
      T.groupby("session").apply(lambda d: tuple(np.round(d.pole_in_time.values[:20], 3))).duplicated().sum(),
      "; identical spike trains", pd.Series([(len(x), float(np.round(x[:3].sum(), 6))) for s in S for x in s.spikes]).duplicated().sum())
# Do lick early trials keep their photostimulation code? Fraction of trials per code, by lick early.
print(pd.crosstab([T.group, T.lick_early], T.stim, normalize="index").round(3))


In [ ]:
# The photostimulation codes. For each code, the percentage of putative pyramidal neurons whose spike
# count differs from control trials (two-tailed t-test, p < 0.01) in 100 ms bins aligned to sample onset
# (pole_in_time): sample 0 to 1.3 s, delay 1.3 to 2.6 s, go cue at 2.6 s.
edges = np.arange(-0.5, 4.0, 0.1)
code_rows = []
for gr in ("fastigial", "dentate", "inhibition"):
    for code in ("1", "2", "3", "4", "5", "6"):
        sig = []
        for p in P:
            if p.group != gr:
                continue
            ctl = (p.t.stim == "0").to_numpy()
            st = (p.t.stim == code).to_numpy()
            if st.sum() < 3:
                continue
            al = p.t.pole_in_time.to_numpy()
            for spk in p.spikes:
                c, C = g.binned(spk, al, edges[0] + 0.05, edges[-1] - 0.05, 0.1, 0.1)
                sig.append(g.ttest_p(C[ctl], C[st]) < 0.01)
        if sig:
            f = 100 * np.mean(sig, 0)
            code_rows.append(pd.Series(f, index=np.round(c, 2), name=f"{gr} code {code} (n={len(sig)})"))
CODES = pd.DataFrame(code_rows)
print(CODES.round(0).astype(int).to_string())
pre = CODES.loc[:, CODES.columns < 0].mean(1)
for name, r in CODES.iterrows():
    first = r.index[np.argmax(r.to_numpy() > 3 * max(pre[name], 1))]
    print(f"{name}: before sample onset {pre[name]:.1f}%; first bin above 3x that level starts at {first - 0.05:.2f} s "
          f"from sample onset")

In [ ]:
# Codes 5 and 6 over the delay: mean absolute rate change (photo-inhibition minus control, spikes/s)
# over the putative pyramidal neurons of the inhibition sessions, 100 ms bins from sample onset; the
# level before sample onset is the floor that noise alone gives an absolute change.
tc = {}
for code in ("5", "6"):
    D_ = []
    for p in P:
        if p.group != "inhibition":
            continue
        ctl = (p.t.stim == "0").to_numpy(); st = (p.t.stim == code).to_numpy()
        if st.sum() < 3:
            continue
        al = p.t.pole_in_time.to_numpy()
        for spk in p.spikes:
            c, C_ = g.binned(spk, al, edges[0] + 0.05, edges[-1] - 0.05, 0.1, 0.1)
            D_.append(np.abs(C_[st].mean(0) - C_[ctl].mean(0)) / 0.1)
    tc[f"code {code}"] = pd.Series(np.mean(D_, 0), index=np.round(c, 2))
TC = pd.DataFrame(tc)
print("floor (bins before sample onset):", TC[TC.index < 0].mean().round(2).to_dict())
print(TC[(TC.index >= 1.25) & (TC.index <= 3.0)].round(2).T.to_string())

In [ ]:
# Behavior by code (trials with a lick, no lick early): fraction correct per group and code.
B = T[(T.lick_early == 0) & T.lick]
print(B.groupby(["group", "stim"]).correct.agg(["mean", "size"]).round(3))
print("fraction of trials per code:", T.groupby("group").stim.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# Clock check, per session: the fraction of neurons changing their rate (p < 0.01) in the 400 ms after the
# onset of delay photostimulation against the 400 ms before it. A response locked to the trial events
# shows only if spikes and trials share a clock; a contrast between conditions, not an overall rate.
ck = []
for p in P:
    codes = g.delay_code(p.group)
    ctl = (p.t.stim == "0").to_numpy()
    st = p.t.stim.isin(codes).to_numpy()
    on = p.t.pole_out_time.to_numpy()
    pre_s = np.array([g.counts(x, on - 0.4, on) for x in p.spikes]).reshape(len(p.spikes), -1)
    post_s = np.array([g.counts(x, on, on + 0.4) for x in p.spikes]).reshape(len(p.spikes), -1)
    fpre = np.mean(g.ttest_p(pre_s[:, ctl], pre_s[:, st], axis=1) < 0.01) if len(p.spikes) else np.nan
    fpost = np.mean(g.ttest_p(post_s[:, ctl], post_s[:, st], axis=1) < 0.01) if len(p.spikes) else np.nan
    # shifted control: the same test with trial labels shifted by one trial
    sh = np.roll(st, 1)
    fsh = np.mean(g.ttest_p(post_s[:, ~sh], post_s[:, sh], axis=1) < 0.01) if len(p.spikes) else np.nan
    ck.append(dict(session=p.path.split("/")[1][:-4], group=p.group, units=len(p.spikes), stim_trials=int(st.sum()),
                   before=fpre, after=fpost, after_labels_shifted=fsh))
CK = pd.DataFrame(ck)
CK["shown"] = (CK.after > CK.before) & (CK.after > CK.after_labels_shifted) & (CK.after > 0)
print(CK.round(3).to_string())
print("sessions where a locked response is shown:", int(CK.shown.sum()), "of", len(CK), "; not shown:",
      CK.loc[~CK.shown, "session"].tolist())

In [ ]:
# Hemispheres: the files against Supplemental Table 1 (recording side from rec_coordinate_ml, negative =
# left; manipulated side from the ogen site).
SUPP = {"BAYLORCD8": ("left ALM", "right CN"), "BAYLORCD12": ("left ALM", "right CN"),
        "BAYLORCD13": ("left ALM", "right CN"), "BAYLORCD14": ("right ALM", "left CN"),
        "BAYLORCD6": ("left ALM", "right FN"), "BAYLORNL15": ("left ALM", "right FN"),
        "BAYLORNL12": ("left ALM", "right DN"), "BAYLORNL14": ("left ALM", "right DN")}
hem = by.reset_index()[["subject", "group", "site", "site_hemisphere", "rec_ml"]].copy()
hem["file_rec"] = np.where(hem.rec_ml.astype(float) < 0, "left ALM", "right ALM")
hem["table_rec"] = hem.subject.map(lambda x: SUPP.get(x, ("?", "?"))[0])
hem["table_manip"] = hem.subject.map(lambda x: SUPP.get(x, ("?", "?"))[1])
hem["recording_contralateral_in_file"] = (hem.file_rec.str[0] != hem.site_hemisphere.str[0])
print(hem.to_string())
print("Supplemental Table 1 names the anm mice AN105 to AN109; the files name them anm00314746 to anm00314760. "
      "By site, anm00314756 (dentate) is AN106 and anm00314757 (right ALM, left FN) is AN107; the other three "
      "(AN105, AN108, AN109; right FN, left ALM) cannot be told apart.")

## Behavior during the recording sessions

Performance (B1) is the fraction correct among trials with a lick, excluding lick early and no lick
trials (Methods, Behavioral data analysis), on the recording sessions, the only behavior in the
dataset. "Contra" and "ipsi" trials are grouped by instructed licking direction relative to the
manipulated hemisphere, taken from the file's ogen site (for the inhibition mice the files and
Supplemental Table 1 disagree; both are run where it matters). Significance is the hierarchical
bootstrap of B1 (10,000 resamples of mice, sessions, trials; one-tailed).

In [ ]:
def behavior_trials(groups, codes, manip=None):
    """Trials with a lick, no lick early, control or one of `codes`, with contra/ipsi side."""
    rows = []
    for p in P:
        if p.group not in groups:
            continue
        t = g.add_trial_fields(S[p.idx].trials)   # every trial: behavior does not need spikes
        t = t[(t.lick_early == 0) & t.lick & t.stim.isin(["0"] + list(codes))].copy()
        side = (manip or {}).get(p.subject, p.hemisphere)
        contra_lick = "L" if side == "right" else "R"
        t["rel"] = np.where(t.instructed == contra_lick, "contra", "ipsi")
        t["photo"] = np.where(t.stim == "0", "control", "stim")
        t["subject"] = p.subject
        t["session"] = p.path
        t["y"] = t.correct.astype(float)
        rows.append(t)
    return pd.concat(rows, ignore_index=True)


def perf_test(df, rng, label, n_boot=N_BOOT):
    obs, boots, pval = g.hierarchical_bootstrap(df, "y", "photo", "control", "stim", n_boot, rng)
    c = df[df.photo == "control"].y.mean()
    s = df[df.photo == "stim"].y.mean()
    return dict(test=label, mice=df.subject.nunique(), sessions=df.session.nunique(),
                n_control=int((df.photo == "control").sum()), n_stim=int((df.photo == "stim").sum()),
                control=round(c, 3), stim=round(s, 3), change=round(obs, 3), p_one_sided=pval)


def per_mouse(df):
    return (df.groupby(["subject", "rel", "photo"]).y.agg(["mean", "size"]).unstack("photo").round(3))

### fig1g-fastigial-activation-early-sample-no-effect

> Fastigial photo-activation during the early sample epoch does not affect behavioral choice for either
> trial type.

Criterion: a claim of absence; performance per trial type, control against early sample
photo-activation (code 1), one-sided bootstrap of B1; reproduced when p > 0.05 for both trial types.

In [ ]:
rng = np.random.default_rng(101)
d1 = behavior_trials(["fastigial"], ["1"])
r = [perf_test(d1[d1.rel == side], rng, f"early sample, {side} trials") for side in ("contra", "ipsi")]
r.append(perf_test(d1, rng, "early sample, pooled"))
R_ES = pd.DataFrame(r)
print(R_ES.to_string())
print(per_mouse(d1))

**Result: reproduced.** On the recording sessions of the 6 fastigial mice, early sample photo-activation (code 1) leaves performance unchanged on contra trials (0.903 against 0.916, one-sided p = 0.31) and ipsi trials (0.842 against 0.824, p = 0.41), both p > 0.05.

### fig1g-fastigial-activation-delay-errors

> Fastigial photo-activation during the early delay epoch lowers behavioral performance, even though the
> photostimulus ends 800 ms before the movement.

Criterion: pooled performance, control against early delay photo-activation (code 2, 0.455 s from delay
onset) in the 6 mice; reproduced when performance drops with one-sided bootstrap p < 0.001.

In [ ]:
rng = np.random.default_rng(102)
d2 = behavior_trials(["fastigial"], ["2"])
R_ED = pd.DataFrame([perf_test(d2, rng, "early delay, pooled")])
print(R_ED.to_string())
print("per mouse:\n", d2.groupby(["subject", "photo"]).y.mean().unstack().round(3))
print("smallest attainable p with 10,000 resamples: 1e-4; resamples with a non-negative change:",
      R_ED.p_one_sided.iloc[0] * N_BOOT)

**Result: reproduced.** Early delay fastigial photo-activation (code 2) lowers pooled performance from 0.873 to 0.630 (change -0.243) in the 6 mice; no bootstrap resample of 10,000 gave a non-negative change (p < 0.0001). Every mouse drops (0.447 to 0.782 with photo-activation against 0.745 to 0.921 in control).

### fig1h-fastigial-activation-delay-bias

> Fastigial photo-activation during the delay epoch biases future movements toward either the
> contralateral or the ipsilateral direction, depending on the mouse.

Criterion: per mouse, the change in performance for contra and ipsi trials, per-mouse one-sided
bootstrap of B1 (sessions and trials of that mouse) at p < 0.05; reproduced when under one reading at
least one mouse has a significant contra drop larger than its ipsi drop and at least one the reverse.
Reading (a), early delay 0.455 s, is the only one in the dataset (reading (b), the 1.255 s whole delay
stimulus, is not in the recording sessions).

In [ ]:
def bias_table(df, rng, n_boot=N_BOOT):
    out = []
    for m, dm in df.groupby("subject"):
        row = dict(subject=m)
        for side in ("contra", "ipsi"):
            ds_ = dm[dm.rel == side]
            obs, _, pv = g.hierarchical_bootstrap(ds_, "y", "photo", "control", "stim", n_boot, rng)
            row[f"{side}_control"] = round(ds_[ds_.photo == "control"].y.mean(), 3)
            row[f"{side}_stim"] = round(ds_[ds_.photo == "stim"].y.mean(), 3)
            row[f"{side}_n_stim"] = int((ds_.photo == "stim").sum())
            row[f"{side}_change"] = round(obs, 3)
            row[f"{side}_p"] = pv
        row["contra_bias_mouse"] = (row["contra_change"] < row["ipsi_change"]) and row["contra_p"] < 0.05 and row["contra_change"] < 0
        row["ipsi_bias_mouse"] = (row["ipsi_change"] < row["contra_change"]) and row["ipsi_p"] < 0.05 and row["ipsi_change"] < 0
        out.append(row)
    return pd.DataFrame(out)


rng = np.random.default_rng(103)
BIAS_ACT = bias_table(d2, rng)
print(BIAS_ACT.to_string())
print("mice with a significant contra drop larger than ipsi:", int(BIAS_ACT.contra_bias_mouse.sum()),
      "; with a significant ipsi drop larger than contra:", int(BIAS_ACT.ipsi_bias_mouse.sum()))
print("at p < 0.01 instead: contra", int(((BIAS_ACT.contra_p < 0.01) & BIAS_ACT.contra_bias_mouse).sum()),
      "ipsi", int(((BIAS_ACT.ipsi_p < 0.01) & BIAS_ACT.ipsi_bias_mouse).sum()))

**Result: reproduced** under reading (a), the only one in the dataset. Two mice (anm00314746, anm00314757) have a significant contra drop larger than the ipsi change (p = 0.023 and 0.001), and three (BAYLORNL15, anm00314758, anm00314760) a significant ipsi drop larger than the contra drop (p < 0.0001, 0.022, < 0.0001). At p < 0.01 one mouse each way remains (1 contra, 2 ipsi).

### fig1g-dentate-activation-little-effect

> Dentate photo-activation produces little effect on behavioral choice.

Criterion: a claim of absence; pooled performance, control against delay dentate photo-activation, the
one-sided bootstrap of B1 at p < 0.05; reproduced when it does not find a drop. Reading (a), the 0.455 s
early delay stimulus (code 4), in the 3 dentate mice recorded here (the paper has 8).

In [ ]:
rng = np.random.default_rng(104)
d4 = behavior_trials(["dentate"], ["4"])
R_DN = pd.DataFrame([perf_test(d4, rng, "dentate early delay, pooled")] +
                    [perf_test(d4[d4.rel == s_], rng, f"dentate early delay, {s_}") for s_ in ("contra", "ipsi")])
print(R_DN.to_string())
print("per mouse:\n", d4.groupby(["subject", "photo"]).y.mean().unstack().round(3))
d3 = behavior_trials(["dentate"], ["3"])
print("dentate early sample (code 3), for reference:\n", pd.DataFrame([perf_test(d3, rng, "dentate early sample")]).to_string())

**Result: reproduced.** In the 3 dentate mice recorded here, early delay dentate photo-activation (code 4) changes pooled performance by -0.038 (0.873 to 0.835), one-sided p = 0.108, so the paper's test finds no drop at p < 0.05 (the paper: p = 0.06 over 8 mice). The ipsi trials alone drop by 0.063 (p = 0.041).

### fig1k-inhibition-delay-bias

> Fastigial photo-inhibition during the delay epoch biases future movements to the contralateral or the
> ipsilateral direction, depending on the mouse.

Criterion: as fig1h-fastigial-activation-delay-bias, on delay photo-inhibition trials. Reading (b), the
1.3 s whole delay photo-inhibition, of CD12, CD13 and CD16 (Extended Data Fig. 2g); CD12 and CD13 are
here, in their recording sessions, with CD08 and CD14 as a variant. Codes 5 and 6 (two powers) are
pooled. The manipulated side is from the files and, as a variant, from Supplemental Table 1.

In [ ]:
rng = np.random.default_rng(105)
d56 = behavior_trials(["inhibition"], ["5", "6"])
BIAS_INH = bias_table(d56, rng)
print("manipulated side from the files:\n", BIAS_INH.to_string())
table_side = {"BAYLORCD8": "right", "BAYLORCD12": "right", "BAYLORCD13": "right", "BAYLORCD14": "left"}
d56t = behavior_trials(["inhibition"], ["5", "6"], manip=table_side)
BIAS_INH_T = bias_table(d56t, np.random.default_rng(106))
print("manipulated side from Supplemental Table 1:\n", BIAS_INH_T.to_string())
for name, tab in (("files", BIAS_INH), ("Supplemental Table 1", BIAS_INH_T)):
    sub = tab[tab.subject.isin(["BAYLORCD12", "BAYLORCD13"])]
    print(f"{name}: CD12 and CD13, contra-bias mice {int(sub.contra_bias_mouse.sum())}, ipsi-bias mice "
          f"{int(sub.ipsi_bias_mouse.sum())}; all 4 mice, contra {int(tab.contra_bias_mouse.sum())}, ipsi {int(tab.ipsi_bias_mouse.sum())}")
print("pooled delay photo-inhibition:", pd.DataFrame([perf_test(d56, rng, "CN photo-inhibition, delay, pooled")]).to_string())

**Result: partially reproduced.** In CD12 and CD13, the two mice of the paper's 1.3 s experiment that are in the dataset, delay photo-inhibition lowers performance on the same trial type in both (drops of 0.161 and 0.221, p = 0.022 and 0.015), contra by the files' hemispheres and ipsi by Supplemental Table 1's; so within those two mice the bias goes one way only. Across the four recorded L7-cre x Ai32 mice both directions occur under either assignment (3 and 1, or 1 and 3). Without the five sessions of the clock check, CD12 and CD13 still drop on the same trial type (section Sensitivity).

### Early licks (for fig1i-fastigial-activation-licking-unaffected, blocked)

The claim also needs lick rates within bouts and bout durations, which need lick times the files do not
hold. The early lick part alone: the fraction of trials with lick_early, control against delay
photo-activation (code 2), per mouse. This does not decide the outcome.

In [ ]:
EL = []
for p in P:
    if p.group != "fastigial":
        continue
    t = g.add_trial_fields(S[p.idx].trials)
    t = t[t.stim.isin(["0", "2"])]
    EL.append(t.assign(subject=p.subject, session=p.path, y=t.lick_early.astype(float),
                       photo=np.where(t.stim == "0", "control", "stim")))
EL = pd.concat(EL)
obs, _, pv = g.hierarchical_bootstrap(EL, "y", "photo", "control", "stim", N_BOOT, np.random.default_rng(107))
print(EL.groupby(["subject", "photo"]).y.mean().unstack().round(3))
print(f"early lick rate control {EL[EL.photo == 'control'].y.mean():.3f}, delay photo-activation "
      f"{EL[EL.photo == 'stim'].y.mean():.3f}, change {obs:+.3f}, one-sided p = {pv:.4f} "
      f"(two-sided {min(1, 2 * pv):.4f}). An early lick restarts the epoch, so a lick during the "
      "photostimulus may change when it is delivered; the files do not say.")

In [ ]:
# fig1l-inhibition-execution-unaffected (blocked): its early lick part, which the files allow. Fraction
# of trials with an early lick, control against delay photo-inhibition (codes 5 and 6) in the 4 L7-cre x
# Ai32 mice (the claim names the 7 mice of Extended Data Fig. 2h, which are not here), bootstrap of B1,
# two-sided at p < 0.05 (the claims file's choice). Lick early trials keep their code (data check).
EL6 = []
for p in P:
    if p.group != "inhibition":
        continue
    t = g.add_trial_fields(S[p.idx].trials)
    t = t[t.stim.isin(["0", "5", "6"])]
    EL6.append(t.assign(subject=p.subject, session=p.path, y=t.lick_early.astype(float),
                        photo=np.where(t.stim == "0", "control", "stim")))
EL6 = pd.concat(EL6)
obs6, _, pv6 = g.hierarchical_bootstrap(EL6, "y", "photo", "control", "stim", N_BOOT, np.random.default_rng(108))
print(EL6.groupby(["subject", "photo"]).y.mean().unstack().round(3))
print(f"early lick fraction control {EL6[EL6.photo == 'control'].y.mean():.3f}, delay photo-inhibition "
      f"{EL6[EL6.photo == 'stim'].y.mean():.3f}, change {obs6:+.3f}, two-sided p = {min(1, 2 * pv6):.4f}")

The early lick part of fig1l-inhibition-execution-unaffected shows no change: 0.109 of control trials
and 0.117 of delay photo-inhibition trials have an early lick (two-sided p = 0.59), in each of the 4 mice
within 0.04. The claim stays blocked: lick rates within bouts need lick times, and the video and jaw
tracking are not in the files.

## ALM selectivity (Fig. 2)

The units of Fig. 2 are the ALM recordings with CN photo-activation (Supplemental Table 1 lists Fig. 2a
and 2c under the fastigial and dentate photo-activation mice): 34 sessions in 9 mice here. Putative
pyramidal neurons by spike width (E1). The selectivity test (E2) uses control trials; the paper does not
say whether photostimulation trials enter, and leaving them out is the standard choice, as they are
perturbed by design (a variant with them is shown).

### fig2-alm-selective-fraction

> Most ALM putative pyramidal neurons (980 of 1194) distinguish "lick left" from "lick right" trials in
> their spike counts.

Criterion: units as E1, test as E2; the fraction is judged (expected 0.82, tolerance 0.082) when the
dataset holds fewer than the 39 sessions of 10 mice, and the counts are reported. Readings: response
window 1.3 or 1.5 s; trials correct only (a) or all trials with a lick (b).

In [ ]:
ACT = [p for p in P if p.group in ("fastigial", "dentate")]
sel_rows = []
SEL = {}
for tr in ("correct", "all"):
    for rw in g.RESPONSE_WINDOWS:
        sel, prep, peri = [], [], []
        for p in ACT:
            s_, ps = g.selective(p, tr, rw)
            sel.append(s_)
            prep.append((ps["sample"] < g.P_SELECT) | (ps["delay"] < g.P_SELECT))
            peri.append(ps[f"response{rw}"] < g.P_SELECT)
        sel, prep, peri = map(np.concatenate, (sel, prep, peri))
        SEL[(tr, rw)] = sel
        sel_rows.append(dict(trials=tr, response_s=rw, neurons=len(sel), selective=int(sel.sum()),
                             fraction=round(sel.mean(), 3), preparatory_of_all=round(prep.mean(), 3),
                             preparatory_of_selective=round(prep.sum() / sel.sum(), 3),
                             peri_movement_of_all=round(peri.mean(), 3),
                             peri_movement_of_selective=round(peri.sum() / sel.sum(), 3)))
SELT = pd.DataFrame(sel_rows)
print(SELT.to_string())
print("expected fraction 0.82 +/- 0.082 (0.738 to 0.902); within:", ((SELT.fraction - 0.82).abs() <= 0.082).tolist())
# per mouse and per group
subj = np.concatenate([[p.subject] * len(p.units) for p in ACT])
grp = np.concatenate([[p.group] * len(p.units) for p in ACT])
print(pd.DataFrame(dict(subject=subj, sel=SEL[("correct", 1.3)])).groupby("subject").sel.agg(["mean", "size"]).round(3))
print(pd.DataFrame(dict(group=grp, sel=SEL[("correct", 1.3)])).groupby("group").sel.agg(["mean", "size"]).round(3))
# variants: the files' cell_type in place of spike width; photostimulation trials included
P_FILE = g.prepare(S, use_file_cell_type=True)
v = np.concatenate([g.selective(p, "correct", 1.3)[0] for p in P_FILE if p.group in ("fastigial", "dentate")])
print(f"variant, files' cell_type 'pyramidal': {v.sum()} of {len(v)} = {v.mean():.3f}")
vv = []
for p in ACT:
    t = p.t
    r = (t.correct & (t.instructed == "R")).to_numpy(); l_ = (t.correct & (t.instructed == "L")).to_numpy()
    s_ = np.zeros(len(p.units), bool)
    for e in ("sample", "delay", "response1.3"):
        s_ |= g.ttest_p(p.ep[e][:, r], p.ep[e][:, l_], axis=1) < g.P_SELECT
    vv.append(s_)
vv = np.concatenate(vv)
print(f"variant, correct trials including photostimulation trials: {vv.sum()} of {len(vv)} = {vv.mean():.3f}")
print("chance level: with three tests at p < 0.05 a non-selective neuron is called selective with probability "
      f"up to {1 - 0.95 ** 3:.3f}")

**Result: reproduced.** Under the correct-trials reading, 800 of 1082 putative pyramidal neurons are selective with a 1.3 s response window (0.739) and 815 (0.753) with 1.5 s, inside 0.738 to 0.902; under the all-trials reading the fraction is 0.734, just outside. The dataset holds 9 of the 10 mice, so the counts (paper 980 of 1194) are reported, not judged. The fraction differs between mice (0.286 to 0.838) and is lower in the fastigial (0.659) than the dentate sessions (0.825).

The ALM fractions of fig2-preparatory-more-in-alm (expected 0.59) and fig2-peri-movement-equal (expected
0.67) are in the table above (`preparatory_*`, `peri_movement_*`). Both claims compare ALM with the CN,
whose recordings are not in the dataset, so they are blocked; the ALM numbers do not decide them.

### fig2cd-selectivity-emerges-gradually

> Selectivity for "lick left" and "lick right" trials emerges gradually in both the ALM and the CN
> population averages, starting during the sample epoch and peaking in the response epoch.

Criterion: population selectivity as E3 in 200 ms windows over the selective neurons; in a region,
reproduced when the bootstrap 95% interval over neurons excludes zero in at least one window of the
sample epoch, the mean over the delay exceeds the mean over the sample, and the maximum lies in the
response epoch. Only ALM is in the dataset. E3 leaves open whether the 10 preference trials are 10 of
each type or 10 in all, and which epoch fixes the preference (the paper says the epoch gave
"qualitatively similar preparatory activity"); every combination is run. Trials: correct control trials (Fig. 2a shows
correct trials; Fig. 2c is not a photostimulation analysis). Time is from the go cue: sample -2.6 to
-1.3 s, delay -1.3 to 0 s, response 0 to 1.3 s. The response epoch's end for the maximum is 1.3 s
(E2 reading (a)); the window centres run to 1.2 s so that no window passes 1.3 s. Neurons are selected (E2) on
the same correct control trials; a variant selects them on one half of the trials and measures the
selectivity on the other half.

In [ ]:
rng = np.random.default_rng(201)
CENT = None
popsel, popsub = {}, {}
sel_mask = SEL[("correct", 1.3)]
k = 0
per_unit = []   # (prep index, unit index within prep)
for p in ACT:
    for j in range(len(p.units)):
        per_unit.append((p, j))
for per_type in (True, False):
    for epoch in ("sample", "delay", "response1.3"):
        curves, subs = [], []
        for (p, j), is_sel in zip(per_unit, sel_mask):
            if not is_sel:
                continue
            t = p.t
            m = ((t.stim == "0") & t.correct).to_numpy()
            rates, prefc = {}, {}
            for side in ("R", "L"):
                mm = m & (t.instructed == side).to_numpy()
                c_, C = g.binned(p.spikes[j], t.cue_time.to_numpy()[mm], -3.0, 1.2, 0.2, 0.05)
                rates[side] = C / 0.2
                prefc[side] = p.ep[epoch][j, mm]
            CENT = c_
            if min(len(prefc["R"]), len(prefc["L"])) < (11 if per_type else 6):
                continue
            _, _, sc = g.pop_selectivity(rates, prefc, rng, g.PREF_TRIALS_ALM, per_type)
            curves.append(sc); subs.append(p.subject)
        popsel[(per_type, epoch)] = np.array(curves)
        popsub[(per_type, epoch)] = np.array(subs)
ep_s = (CENT >= -2.6 + 0.1) & (CENT <= -1.3 - 0.1)
ep_d = (CENT >= -1.3 + 0.1) & (CENT <= -0.1)
ep_r = (CENT >= 0.1) & (CENT <= 1.2)
def fig2c_table(curves):
    rows = []
    for (per_type, epoch), X in curves.items():
        mean, lo, hi, _ = g.boot_mean_ci(X, rng, N_CI)
        imax = int(np.argmax(mean))
        rows.append(dict(pref_trials="10 of each type" if per_type else "10 in all", pref_epoch=epoch, neurons=len(X),
                         sample_windows_ci_above_0=int((lo[ep_s] > 0).sum()), sample_mean=round(mean[ep_s].mean(), 2),
                         delay_mean=round(mean[ep_d].mean(), 2), response_mean=round(mean[ep_r].mean(), 2),
                         max_at_s=round(CENT[imax], 2), max_value=round(mean[imax], 2),
                         pre_sample_mean=round(mean[CENT <= -2.7].mean(), 2),
                         reproduced=bool((lo[ep_s] > 0).any() and mean[ep_d].mean() > mean[ep_s].mean() and ep_r[imax])))
    return pd.DataFrame(rows)


FIG2C = fig2c_table(popsel)
print("selective neurons:", int(sel_mask.sum()), "; window centres", CENT[0], "to", CENT[-1], "s from the go cue")
print(FIG2C.to_string())
fig, ax = plt.subplots(figsize=(6, 3))
for (per_type, epoch), X in popsel.items():
    ax.plot(CENT, X.mean(0), label=f"{'10 each' if per_type else '10 all'}, {epoch}")
for x in (-2.6, -1.3, 0):
    ax.axvline(x, color="k", ls="--", lw=0.5)
ax.set_xlabel("time from go cue (s)"); ax.set_ylabel("selectivity (spikes/s)"); ax.legend(fontsize=6)
print("plotted range:", round(min(X.mean(0).min() for X in popsel.values()), 2), "to",
      round(max(X.mean(0).max() for X in popsel.values()), 2), "spikes/s; none outside the axes")
# per mouse (main reading: 10 of each type, delay epoch)
X = popsel[(True, "delay")]
print("selective neurons left out for too few trials to set the preference:",
      {f"{'10 each' if k_[0] else '10 all'}, {k_[1]}": int(sel_mask.sum()) - len(v_) for k_, v_ in popsel.items()})
if True:
    pm = pd.DataFrame(dict(subject=popsub[(True, "delay")], sample=X[:, ep_s].mean(1), delay=X[:, ep_d].mean(1), response=X[:, ep_r].mean(1)))
    print(pm.groupby("subject").agg(["mean", "size"]).round(2))
# Variant: neurons selected by E2 on a random half of the correct control trials of each type, and the
# population selectivity (preference and held-out trials) computed on the other half only, so that the
# selection and the sample-epoch test share no trials. Selecting on the same trials biases that test
# above 0.
rng_h = np.random.default_rng(202)
held = {k: [] for k in popsel}
n_sel_half = 0
for p in ACT:
    t = p.t
    m = ((t.stim == "0") & t.correct).to_numpy()
    half = np.zeros(len(t), bool)
    for side in ("R", "L"):
        idx = np.where(m & (t.instructed == side).to_numpy())[0]
        half[rng_h.choice(idx, len(idx) // 2, replace=False)] = True
    other = m & ~half
    sel_half, _ = g.selective(p, "correct", 1.3, mask=half)
    n_sel_half += int(sel_half.sum())
    for j in np.where(sel_half)[0]:
        rates, mm = {}, {}
        for side in ("R", "L"):
            mm[side] = other & (t.instructed == side).to_numpy()
            rates[side] = g.binned(p.spikes[j], t.cue_time.to_numpy()[mm[side]], -3.0, 1.2, 0.2, 0.05)[1] / 0.2
        for per_type in (True, False):
            if min(mm["R"].sum(), mm["L"].sum()) < (11 if per_type else 6):
                continue
            for epoch in ("sample", "delay", "response1.3"):
                prefc = {side: p.ep[epoch][j, mm[side]] for side in ("R", "L")}
                held[(per_type, epoch)].append(g.pop_selectivity(rates, prefc, rng_h, g.PREF_TRIALS_ALM, per_type)[2])
held = {k: np.array(v) for k, v in held.items()}
print("variant, selection on one half of the trials and selectivity on the other:", n_sel_half, "selective neurons")
print(fig2c_table(held).to_string())


**Result: partially reproduced.** On ALM, every reading has population selectivity above zero during the sample epoch (16 to 22 windows with the interval above 0) and a delay mean above the sample mean (0.66 to 1.76 against 0.22 to 0.60 spikes/s), so selectivity emerges in the sample epoch and grows. The maximum lies in the response epoch only when the response epoch fixes each neuron's preferred trial type (0.50 s and 0.25 s after the go cue); with the sample or delay epoch it lies before the go cue (-0.60 to -0.05 s). The paper says the epoch gave "qualitatively similar preparatory activity", and here too the preparatory part (sample-epoch windows above 0, delay mean above sample mean) is similar under every epoch; that sentence does not cover the peak's place, which holds under one of the three epochs only. Selecting the neurons on one half of the trials and measuring on the other gives the same pattern (13 to 22 sample windows above 0; maximum in the response epoch only with the response epoch). The CN half of the claim is not in the dataset.

## Photostimulation effects on ALM neurons (Figs. 3 and 4)

E4: lick left and lick right trials pooled; two-tailed t-test on spike counts, control against
photostimulation trials, at p < 0.01. Trials: readings all trials with a lick (correct and error) and
correct trials only. E5 latency: per modulated neuron the rate change (photostimulation minus control),
sign flipped for suppressed neurons (sign of the change in the window that selected it), in 10 ms bins
in 1 ms steps around photostimulus onset (delay onset); onset is the first bin where the mean over
neurons is above 0 by a two-tailed one-sample t-test at p < 0.01; neurons resampled 10,000 times and the
mean onset reported. The paper does not say which time a 10 ms bin is labelled with; the three labels
(bin start, centre, end) are run, as they shift the latency by 5 to 10 ms against tolerances of 2.6 to
5.0 ms. The search starts at the first bin whose label is at or after onset.

In [ ]:
LAT_STARTS = np.round(np.arange(-0.050, 0.100, 0.001), 3)   # bin starts, s from photostimulus onset


def modulation(group, codes, window, trials="all"):
    """E4 per putative pyramidal neuron of `group` sessions: p-value and rate change (spikes/s) in
    `window` (an epoch name), control against `codes`; also the per-neuron fine time course for E5."""
    out = []
    for p in P:
        if p.group != group:
            continue
        t = p.t
        base = t.correct if trials == "correct" else t.lick
        ctl = (base & (t.stim == "0")).to_numpy()
        st = (base & t.stim.isin(codes)).to_numpy()
        a, b = g.epoch_windows(t)[window]
        dur = (b - a)[0]
        on = t.pole_out_time.to_numpy()
        for j, spk in enumerate(p.spikes):
            c = p.ep[window][j]
            pv = g.ttest_p(c[ctl], c[st])
            lo = on[:, None] + LAT_STARTS[None, :]
            fine = (np.searchsorted(spk, (lo + 0.010).ravel()) - np.searchsorted(spk, lo.ravel())).reshape(lo.shape) / 0.010
            out.append(dict(subject=p.subject, session=p.path, unit=j, p=float(pv),
                            ctl_rate=c[ctl].mean() / dur, stim_rate=c[st].mean() / dur,
                            n_ctl=int(ctl.sum()), n_stim=int(st.sum()),
                            fine=fine[st].mean(0) - fine[ctl].mean(0)))
    df = pd.DataFrame(out)
    df["change"] = df.stim_rate - df.ctl_rate
    df["modulated"] = df.p < g.P_PHOTO
    return df


def latency(df, rng, n_boot=N_BOOT, baseline=False):
    """E5. baseline=True is a variant (not in the paper): each neuron's mean change in the bins ending
    before onset (starts -50 to -11 ms) is subtracted first, as a sign-flipped change is biased above 0
    wherever a neuron's rate differs between conditions before the photostimulus."""
    m = df[df.modulated]
    delta = np.vstack(m.fine.to_numpy()) * np.sign(m.change.to_numpy())[:, None]
    if baseline:
        delta = delta - delta[:, LAT_STARTS <= -0.011].mean(1, keepdims=True)
    res = {}
    for label, shift in (("start", 0.0), ("centre", 5.0), ("end", 10.0)):
        cms = np.round(LAT_STARTS * 1000) + shift
        obs, bmean, bsd, nnan = g.onset_latency(delta, cms, g.P_PHOTO, n_boot, rng, search_from_ms=0.0)
        res[label] = dict(observed_ms=obs, boot_mean_ms=round(bmean, 2), boot_sd_ms=round(bsd, 2), no_onset=nnan)
    return pd.DataFrame(res).T, delta


def summarize_mod(df, expected_n, expected_k):
    n, k = len(df), int(df.modulated.sum())
    inc, dec = int((df.modulated & (df.change > 0)).sum()), int((df.modulated & (df.change < 0)).sum())
    return dict(neurons=n, modulated=k, fraction=round(k / n, 3), increase=inc, decrease=dec,
                expected_neurons=expected_n, expected_modulated=expected_k,
                expected_fraction=round(expected_k / expected_n, 3),
                chance_fraction=g.P_PHOTO)

### fig3d-alm-altered-by-cn-inhibition

> CN photo-inhibition during the delay rapidly alters the activity of many ALM neurons (197 of 454),
> with a latency of 7.1 plus or minus 3.8 ms.

Criterion: ALM recordings during CN photo-inhibition in the 4 L7-cre x Ai32 mice; photo-inhibition 1.3 s
from delay onset, both powers pooled (codes 5 and 6); modulated as E4 on the whole delay epoch; latency
as E5 (both readings of the window, which coincide here: the photostimulus fills the delay epoch).
Reproduced when under one reading the count (or the fraction 0.43 where the dataset holds fewer neurons)
and the latency are within tolerance (count 197 +/- 19.7, latency 7.1 +/- 3.8 ms).

In [ ]:
rng = np.random.default_rng(301)
MOD_INH = {tr: modulation("inhibition", ["5", "6"], "delay", tr) for tr in ("all", "correct")}
print(pd.DataFrame({tr: summarize_mod(d, 454, 197) for tr, d in MOD_INH.items()}).T.to_string())
LAT_INH = {}
for tr, d in MOD_INH.items():
    LAT_INH[tr], DELTA_INH = latency(d, rng)
    print(f"trials {tr}: latency (ms) by bin label\n", LAT_INH[tr].to_string())
    print("  variant, pre-onset change subtracted:\n", latency(d, rng, n_boot=1000, baseline=True)[0].to_string())
print("count tolerance 177.3 to 216.7; fraction tolerance (10% of 0.434) 0.39 to 0.48; latency 3.3 to 10.9 ms")
print("per mouse (all trials):\n", MOD_INH["all"].groupby("subject").modulated.agg(["sum", "size", "mean"]).round(3))
print("per code, all trials: ", {c: summarize_mod(modulation("inhibition", [c], "delay"), 454, 197)["fraction"] for c in ("5", "6")})
d = DELTA_INH
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.plot(LAT_STARTS * 1000, d.mean(0)); ax.axvline(0, color="k", lw=0.5)
ax.set_xlabel("bin start from photo-inhibition onset (ms)"); ax.set_ylabel("|rate change| (spikes/s)")
print("plotted range", round(d.mean(0).min(), 2), "to", round(d.mean(0).max(), 2))

In [ ]:
# When the ALM change starts, per code: mean sign-flipped rate change (spikes/s) over the neurons modulated
# in the delay epoch (E4, all trials), 10 ms bins labelled by their start, every 2 ms from 10 ms before to
# 40 ms after delay onset (pole_out_time), and the one-sample t-test p of E5 (p < 0.01 marked *). The
# files hold no laser trace, so where the photostimulus starts relative to pole_out_time (for photo-
# inhibition, also the phase at which the 40 Hz sinusoid starts) can only be read from this response.
show = (LAT_STARTS >= -0.010 - 1e-9) & (LAT_STARTS <= 0.040 + 1e-9) & (np.round(LAT_STARTS * 1000) % 2 == 0)
ONSET = {}
for gr, code in (("inhibition", "5"), ("inhibition", "6"), ("fastigial", "2"), ("dentate", "4")):
    d_ = modulation(gr, [code], "delay")
    m_ = d_[d_.modulated]
    dl = np.vstack(m_.fine.to_numpy()) * np.sign(m_.change.to_numpy())[:, None]
    pv = stats.ttest_1samp(dl, 0.0, axis=0).pvalue
    ONSET[f"{gr} {code} (n={len(m_)})"] = [f"{v:.1f}{'*' if q < 0.01 else ''}" for v, q in zip(dl.mean(0)[show], pv[show])]
print(pd.DataFrame(ONSET, index=np.round(LAT_STARTS[show] * 1000).astype(int)).T.to_string())

**Result: not reproduced.** 270 of 676 putative pyramidal neurons change their delay rate with CN photo-inhibition (all trials; 223 with correct trials only), against 197 +/- 19.7 of 454. The dataset holds more neurons than the paper, so the criterion's fraction, given for fewer neurons, does not apply; the fraction, 0.399, is inside 0.39 to 0.48. The latency is longer than the paper's under every bin label and trial reading: bootstrap means of 14.1 to 22.3 ms against 7.1 +/- 3.8 ms, with a bootstrap s.d. of 11.5 to 14.7 ms (paper 3.8), and observed onsets of 24 to 37 ms. Per code, the population change becomes significant and stays so from 30 ms (code 5) and 32 ms (code 6) after delay onset, where fastigial photo-activation gives 4 ms (table above); code 6 also has a small significant change in the bins starting -2 to 2 ms, gone by 4 ms; the files hold no laser trace, so a photostimulus that starts after `pole_out_time` cannot be ruled out. Without the five sessions of the clock check, the bin-start bootstrap mean is 9.9 ms (s.d. 11.4) while the observed onset stays at 25 ms; the bootstrap mean then mixes resamples that find an early bump at delay onset with ones that find the change near 25 ms (section Sensitivity).

### fig3e-alm-mean-rate-unchanged

> Despite the changes in individual neurons, the average ALM spike rate is nearly unchanged by CN
> photo-inhibition (4.15 against 4.39 spikes/s).

Criterion: a claim of absence, the paper's two-tailed t-test at p < 0.05; mean delay epoch rate per
neuron in control and photo-inhibition trials (E4 trials). Run under each reading: neurons all (a) or the
modulated (b); t-test paired over neurons or unpaired. Reproduced when the rates are within tolerance
under one reading of the neurons (4.15 +/- 0.415, 4.39 +/- 0.439) and the t-test finds no difference.

In [ ]:
rows = []
for tr, d in MOD_INH.items():
    for nn, dd in (("all", d), ("modulated", d[d.modulated])):
        rows.append(dict(trials=tr, neurons=nn, n=len(dd), control=round(dd.ctl_rate.mean(), 3),
                         photo_inhibition=round(dd.stim_rate.mean(), 3), change=round(dd.change.mean(), 3),
                         p_paired=stats.ttest_rel(dd.ctl_rate, dd.stim_rate).pvalue,
                         p_unpaired=stats.ttest_ind(dd.ctl_rate, dd.stim_rate).pvalue))
R3E = pd.DataFrame(rows)
R3E["within_tol"] = ((R3E.control - 4.15).abs() <= 0.415) & ((R3E.photo_inhibition - 4.39).abs() <= 0.439)
print(R3E.to_string())
print("per mouse, all neurons, all trials:\n", MOD_INH["all"].groupby("subject")[["ctl_rate", "stim_rate"]].mean().round(2))

**Result: partially reproduced.** The mean delay rate over all 676 neurons changes little (3.379 to 3.767 spikes/s, +0.387), and the unpaired t-test finds no difference (p = 0.214); the paired test does (p = 0.022). Correct trials give the same (3.366 and 3.728; unpaired p = 0.246, paired p = 0.029). The control rate is about 19% below the paper's 4.15 spikes/s and the photo-inhibition rate about 14% below its 4.39, both outside tolerance under every reading; the modulated neurons give 4.652 and 5.780 (unpaired p = 0.069), both above their tolerances. The dataset holds more slow neurons than the paper's count allows (data check: the 454 fastest have a mean of 4.89 spikes/s), so the rate level depends on a neuron selection the paper does not describe. Assumed: the absence test is the unpaired test, as the paired and unpaired tests disagree and the paper's p = 0.58 for a 0.24 spikes/s change fits an unpaired test over neurons; the outcome rests on that choice.

### fig3e-alm-increase-decrease

> Among ALM neurons modulated by CN photo-inhibition, 108 increase and 89 decrease their responses.

Criterion: sign of the delay epoch rate change among the neurons of fig3d; both counts within
tolerance (108 +/- 10.8, 89 +/- 8.9) or, where the dataset holds fewer neurons, the increase fraction
(0.55) within 10% (0.495 to 0.605).

In [ ]:
for tr, d in MOD_INH.items():
    m = d[d.modulated]
    inc, dec = int((m.change > 0).sum()), int((m.change < 0).sum())
    print(f"trials {tr}: increase {inc}, decrease {dec}, increase fraction {inc / len(m):.3f}; "
          f"binomial two-sided p = {stats.binomtest(inc, len(m)).pvalue:.3g}")
    print(m.assign(up=m.change > 0).groupby("subject").up.agg(["sum", "size"]).T.to_string())

**Result: partially reproduced.** Of the 270 modulated neurons (all trials), 158 increase and 112 decrease, both outside the tolerances of 108 +/- 10.8 and 89 +/- 8.9. The dataset holds more neurons than the paper, so the criterion's fraction, given for fewer neurons, does not strictly apply; the increase fraction 0.585 is inside 0.495 to 0.605. With correct trials only the counts are 137 and 86 (0.614, just outside). Increases outnumber decreases, as in the paper, with the same proportion but not the same counts.

### fig3f-alm-selectivity-abolished-by-cn-inhibition

> CN photo-inhibition abolishes ALM trial-type selectivity.

Criterion: population selectivity as E3 (preferred type from 10 trials) over selective ALM neurons tested
on > 3 trials in all conditions (n = 329 in the paper), control against CN photo-inhibition trials, all
trials grouped by instructed type; windows (a) the whole delay epoch, (b) its last 500 ms. Reproduced
when under one reading the control selectivity is above zero (bootstrap 95% interval over neurons) and
the photo-inhibition selectivity's interval includes zero. Selective: E2 on control trials (correct
trials, response window 1.3 s; the other readings are shown as counts). The 10 preference trials are 5
of each type, from control trials (open in E3; 10 of each type is run as a variant); they are left out
of the control selectivity. The preference trials
are a random draw, so each reading is repeated over 20 draws and judged by how many meet the criterion.


In [ ]:
def photo_selectivity(group, codes, windows, rng, per_type=False, sel_reading=("correct", 1.3)):
    """E3 for photostimulation: returns per window a table of control and stim selectivity per neuron."""
    res = {w: [] for w in windows}
    n_sel = 0
    for p in P:
        if p.group != group:
            continue
        sel, _ = g.selective(p, *sel_reading)
        t = p.t
        lick = t.lick.to_numpy()
        ctl = lick & (t.stim == "0").to_numpy()
        st = lick & t.stim.isin(codes).to_numpy()
        R = (t.instructed == "R").to_numpy(); L = (t.instructed == "L").to_numpy()
        ok = min((ctl & R).sum(), (ctl & L).sum(), (st & R).sum(), (st & L).sum()) > 3
        for j in np.where(sel)[0]:
            n_sel += 1
            if not ok:
                continue
            k = g.PREF_TRIALS_ALM if per_type else g.PREF_TRIALS_ALM // 2
            iR, iL = np.where(ctl & R)[0], np.where(ctl & L)[0]
            if min(len(iR), len(iL)) - k < 2:
                continue
            uR, uL = rng.choice(iR, k, replace=False), rng.choice(iL, k, replace=False)
            for w in windows:
                a, b = g.epoch_windows(t)[w]
                rate = p.ep[w][j] / (b - a)
                pref = "R" if rate[uR].mean() >= rate[uL].mean() else "L"
                kR, kL = np.setdiff1d(iR, uR), np.setdiff1d(iL, uL)
                sgn = 1 if pref == "R" else -1
                res[w].append(dict(subject=p.subject, ctl=sgn * (rate[kR].mean() - rate[kL].mean()),
                                   stim=sgn * (rate[st & R].mean() - rate[st & L].mean()),
                                   p_ctl=float(g.ttest_p(rate[ctl & R], rate[ctl & L])),
                                   p_stim=float(g.ttest_p(rate[st & R], rate[st & L])),
                                   ctl_raw_sgn=np.sign(rate[ctl & R].mean() - rate[ctl & L].mean()),
                                   stim_raw_sgn=np.sign(rate[st & R].mean() - rate[st & L].mean())))
    return {w: pd.DataFrame(v) for w, v in res.items()}, n_sel


def ci_row(x, rng):
    m, lo, hi, _ = g.boot_mean_ci(np.asarray(x)[:, None], rng, N_CI)
    return round(float(m[0]), 3), round(float(lo[0]), 3), round(float(hi[0]), 3)

In [ ]:
rng = np.random.default_rng(302)
rows = []
for per_type in (False, True):
    PS3, nsel = photo_selectivity("inhibition", ["5", "6"], ["delay", "delay_last500"], rng, per_type)
    for w, d in PS3.items():
        c, s_ = ci_row(d.ctl, rng), ci_row(d.stim, rng)
        rows.append(dict(pref="5 of each" if not per_type else "10 of each", window=w, selective=nsel, neurons=len(d),
                         control=c[0], control_ci=c[1:], photo=s_[0], photo_ci=s_[1:],
                         control_above_0=c[1] > 0, photo_includes_0=s_[1] <= 0 <= s_[2],
                         reproduced=(c[1] > 0) and (s_[1] <= 0 <= s_[2])))
    if not per_type:
        PS3_MAIN = PS3
R3F = pd.DataFrame(rows)
print(R3F.to_string())
# The preference trials are a random draw, so each reading is repeated over 20 draws: how many meet the
# criterion, and the range of the photo-inhibition selectivity and of its interval's bounds.
rep = []
for seed in range(20):
    r_ = np.random.default_rng(3020 + seed)
    for per_type in (False, True):
        PSx, _ = photo_selectivity("inhibition", ["5", "6"], ["delay", "delay_last500"], r_, per_type)
        for w, d in PSx.items():
            c, s_ = ci_row(d.ctl, r_), ci_row(d.stim, r_)
            rep.append(dict(pref="5 of each" if not per_type else "10 of each", window=w, seed=seed, photo=s_[0],
                            photo_lo=s_[1], photo_hi=s_[2], control_lo=c[1], reproduced=(c[1] > 0) and (s_[1] <= 0 <= s_[2])))
REP3F = pd.DataFrame(rep)
print(REP3F.groupby(["pref", "window"]).agg(draws=("seed", "size"), reproduced=("reproduced", "sum"),
                                            photo_min=("photo", "min"), photo_max=("photo", "max"),
                                            photo_lo_min=("photo_lo", "min"), photo_lo_max=("photo_lo", "max"),
                                            control_lo_min=("control_lo", "min")).round(3).to_string())
for tr in ("correct", "all"):
    for rw in g.RESPONSE_WINDOWS:
        print(f"selective inhibition-session neurons under E2 ({tr}, {rw} s):",
              int(sum(g.selective(p, tr, rw)[0].sum() for p in P if p.group == "inhibition")))
print("per mouse, whole delay:\n", PS3_MAIN["delay"].groupby("subject")[["ctl", "stim"]].agg(["mean", "size"]).round(2))

**Result: reproduced.** Over 438 selective neurons tested on more than 3 trials in all conditions (456 selective; paper 329), the last 500 ms of the delay meet the criterion in most draws of the preference trials: 15 of 20 with 5 of each type and 13 of 20 with 10 of each type (the first draw: control 1.055 spikes/s, interval 0.749 to 1.405; photo-inhibition 0.006, -0.46 to 0.415). Over the whole delay the photo-inhibition selectivity is reduced but more often above 0 (11 and 6 of 20 draws meet the criterion). In the last 500 ms the photo-inhibition selectivity varies widely between draws (-0.179 to 0.611 spikes/s with 5 of each type, -0.076 to 0.695 with 10), so a single draw does not settle it. The neuron count, 438, is outside 329 +/- 32.9. Without the five sessions of the clock check, 16 of 20 draws meet the criterion in the last 500 ms (section Sensitivity).

### fig4-fastigial-activation-alm-latency

> Fastigial photo-activation changes ALM activity rapidly (latency 4.5 plus or minus 2.6 ms), consistent
> with a disynaptic pathway through the thalamus.

Criterion: ALM pyramidal neurons in the fastigial sessions (n = 446 in the paper), early delay
photo-activation (code 2), powers pooled; modulated as E4 at p < 0.01; latency as E5, both readings of
the window that decides "modulated" ((a) the photostimulation period, 0.455 s from delay onset; (b) the
whole delay epoch). Reproduced when under one reading the latency (4.5 +/- 2.6 ms) and the count (227 +/-
22.7, or the fraction 0.51 where the dataset holds fewer neurons) are within tolerance.

In [ ]:
rng = np.random.default_rng(401)
MOD_FN, LAT_FN = {}, {}
rows = []
for win in ("act_period", "delay"):
    for tr in ("all", "correct"):
        d = modulation("fastigial", ["2"], win, tr)
        MOD_FN[(win, tr)] = d
        lat, _ = latency(d, rng)
        LAT_FN[(win, tr)] = lat
        s_ = summarize_mod(d, 446, 227)
        rows.append(dict(window=win, trials=tr, **{k: s_[k] for k in ("neurons", "modulated", "fraction", "increase", "decrease")},
                         **{f"latency_{k}": lat.loc[k, "boot_mean_ms"] for k in lat.index},
                         **{f"observed_{k}": lat.loc[k, "observed_ms"] for k in lat.index}))
R4 = pd.DataFrame(rows)
print(R4.to_string())
for key in (("act_period", "all"), ("delay", "all")):
    print(key, "variant, pre-onset change subtracted:\n", latency(MOD_FN[key], rng, n_boot=1000, baseline=True)[0].to_string())
print("tolerances: count 204.3 to 249.7, fraction (10% of 0.509) 0.458 to 0.560, latency 1.9 to 7.1 ms")
print("per mouse, photostimulation period, all trials:\n",
      MOD_FN[("act_period", "all")].groupby("subject").modulated.agg(["sum", "size", "mean"]).round(3))

**Result: partially reproduced.** The latency matches: with the photostimulation-period window and all trials, the bootstrap mean onset is 2.07 ms by bin start and 5.60 ms by bin centre, inside 4.5 +/- 2.6 ms (10.25 ms by bin end). Fewer neurons are modulated: 162 of 557 (0.291) in the photostimulation period and 178 (0.320) over the delay, against 0.51 in the paper (0.458 to 0.560). The per-code time course (after fig3d) shows the population change significant from the bin starting 4 ms after delay onset.

### fig4g-dentate-effect-recovers (latency and count; the recovery is below, after fig4f)

Criterion: as fig4-fastigial-activation-alm-latency on the dentate sessions (n = 466 in the paper):
latency 8.3 +/- 5.0 ms, count 163 +/- 16.3 or fraction 0.35. The mean rate change before onset is shown,
as a sign-flipped change is biased away from 0 wherever a neuron's rate differs between conditions.

In [ ]:
rng = np.random.default_rng(402)
MOD_DN, rows = {}, []
for win in ("act_period", "delay"):
    for tr in ("all", "correct"):
        d = modulation("dentate", ["4"], win, tr)
        MOD_DN[(win, tr)] = d
        lat, delta = latency(d, rng)
        s_ = summarize_mod(d, 466, 163)
        pre = delta[:, LAT_STARTS < -0.010].mean()
        rows.append(dict(window=win, trials=tr, **{k: s_[k] for k in ("neurons", "modulated", "fraction")},
                         **{f"latency_{k}": lat.loc[k, "boot_mean_ms"] for k in lat.index},
                         **{f"observed_{k}": lat.loc[k, "observed_ms"] for k in lat.index},
                         pre_onset_change=round(pre, 2)))
R4G = pd.DataFrame(rows)
print(R4G.to_string())
for key in (("act_period", "all"), ("delay", "all")):
    print(key, "variant, pre-onset change subtracted:\n", latency(MOD_DN[key], rng, n_boot=1000, baseline=True)[0].to_string())
print("tolerances: count 146.7 to 179.3, fraction (10% of 0.35) 0.315 to 0.385, latency 3.3 to 13.3 ms")
print("per mouse:\n", MOD_DN[("act_period", "all")].groupby("subject").modulated.agg(["sum", "size", "mean"]).round(3))

### fig4f-fastigial-effect-persists

> Transient fastigial photo-activation alters ALM preparatory activity long (more than 1 s) after the
> photostimulus ends.

Criterion: for each ALM neuron, E4 in 400 ms bins in 50 ms steps gives the fraction with a significant
change (p < 0.01); compared with (a) the 1% expected by chance, one-sided binomial test, or (b) the
fraction in the bins before photostimulus onset, a one-sided test of two proportions; reproduced when
under one reading the fraction in bins starting 1 s or more after photostimulus offset (0.455 s after
delay onset), and ending before the go cue or within the response epoch, exceeds the comparison at p <
0.05. Bins are aligned to delay onset and start at multiples of 50 ms; the qualifying bins start 1.5 to
2.2 s after delay onset (the response epoch ends 2.6 s after delay onset under the 1.3 s reading). Each
qualifying bin is tested, and the outcome uses the first (the bin nearest to 1 s after offset) and all of
them. The pre-onset bins are those ending at or before delay onset, from 1.3 s before it (the sample
epoch). Trials: all trials with a lick (E4 reading a), correct trials as a variant.

In [ ]:
BIN_STARTS = np.round(np.arange(-1.3, 2.6 - 0.4 + 1e-9, 0.05), 3)


def bin_fraction(group, codes, trials="all"):
    sig = []
    for p in P:
        if p.group != group:
            continue
        t = p.t
        base = t.correct if trials == "correct" else t.lick
        ctl = (base & (t.stim == "0")).to_numpy()
        st = (base & t.stim.isin(codes)).to_numpy()
        on = t.pole_out_time.to_numpy()
        lo = on[:, None] + BIN_STARTS[None, :]
        for spk in p.spikes:
            C = (np.searchsorted(spk, (lo + 0.4).ravel()) - np.searchsorted(spk, lo.ravel())).reshape(lo.shape)
            sig.append(g.ttest_p(C[ctl], C[st]) < g.P_PHOTO)
    return np.array(sig)


def late_tests(sig, offset=g.ACTIVATION_S):
    n = len(sig)
    frac = sig.mean(0)
    pre = BIN_STARTS + 0.4 <= 1e-9
    k_pre, n_pre = int(sig[:, pre].sum()), int(sig[:, pre].size)
    late = BIN_STARTS >= offset + 1.0 - 1e-9
    # the pre-onset bins overlap, so they are not independent samples; the last of them alone (0.4 s before
    # onset) is a comparison without that
    b1 = int(np.argmin(np.abs(BIN_STARTS + 0.4)))
    k1 = int(sig[:, b1].sum())
    rows = []
    for b in np.where(late)[0]:
        k = int(sig[:, b].sum())
        pa = stats.binomtest(k, n, 0.01, alternative="greater").pvalue
        # two proportions, one-sided: late bin against the pooled pre-onset bins
        p1, p2 = k / n, k_pre / n_pre
        pp = (k + k_pre) / (n + n_pre)
        z = (p1 - p2) / np.sqrt(pp * (1 - pp) * (1 / n + 1 / n_pre)) if 0 < pp < 1 else 0.0
        pb = stats.norm.sf(z)
        pp1 = (k + k1) / (2 * n)
        z1 = (p1 - k1 / n) / np.sqrt(pp1 * (1 - pp1) * 2 / n) if 0 < pp1 < 1 else 0.0
        rows.append(dict(bin_start_from_delay=BIN_STARTS[b], bin_start_from_offset=round(BIN_STARTS[b] - offset, 3),
                         significant=k, fraction=round(p1, 4), p_vs_chance=pa, p_vs_pre=pb,
                         p_vs_last_pre_bin=stats.norm.sf(z1)))
    return pd.DataFrame(rows), k_pre / n_pre, frac


rng = np.random.default_rng(403)
SIG_FN = {tr: bin_fraction("fastigial", ["2"], tr) for tr in ("all", "correct")}
SIG_DN = {tr: bin_fraction("dentate", ["4"], tr) for tr in ("all", "correct")}


def bin_fraction_by_type(group, codes):
    """Variant: correct trials, lick right and lick left tested apart; significant when either p < 0.005,
    so that a change in the mix of lick directions cannot make a neuron significant (chance about 1%)."""
    sig = []
    for p in P:
        if p.group != group:
            continue
        t = p.t
        ctl = (t.correct & (t.stim == "0")).to_numpy()
        st = (t.correct & t.stim.isin(codes)).to_numpy()
        R = (t.instructed == "R").to_numpy()
        lo = t.pole_out_time.to_numpy()[:, None] + BIN_STARTS[None, :]
        for spk in p.spikes:
            C_ = (np.searchsorted(spk, (lo + 0.4).ravel()) - np.searchsorted(spk, lo.ravel())).reshape(lo.shape)
            pmin = np.minimum(g.ttest_p(C_[ctl & R], C_[st & R]), g.ttest_p(C_[ctl & ~R], C_[st & ~R]))
            sig.append(pmin < g.P_PHOTO / 2)
    return np.array(sig)


SIG_FN_TYPE = bin_fraction_by_type("fastigial", ["2"])
SIG_DN_TYPE = bin_fraction_by_type("dentate", ["4"])

In [ ]:
for tr in ("all", "correct"):
    LT, fpre, frac = late_tests(SIG_FN[tr])
    print(f"fastigial, trials {tr}: neurons {len(SIG_FN[tr])}; pre-onset fraction {fpre:.4f}")
    print(LT.to_string())
    print("first qualifying bin exceeds (a) chance:", LT.p_vs_chance.iloc[0] < 0.05, "; (b) pre-onset:", LT.p_vs_pre.iloc[0] < 0.05,
          "| all qualifying bins exceed (a):", bool((LT.p_vs_chance < 0.05).all()), "; (b):", bool((LT.p_vs_pre < 0.05).all()))
LT_FN, FPRE_FN, FRAC_FN = late_tests(SIG_FN["all"])
LTt, fpre_t, _ = late_tests(SIG_FN_TYPE)
print(f"variant within trial type (correct trials): pre-onset fraction {fpre_t:.4f}; qualifying bins {LTt.fraction.min():.4f} "
      f"to {LTt.fraction.max():.4f}; all exceed (a): {bool((LTt.p_vs_chance < 0.05).all())}, (b): {bool((LTt.p_vs_pre < 0.05).all())}, "
      f"last pre-onset bin: {bool((LTt.p_vs_last_pre_bin < 0.05).all())}")
print("every qualifying bin starts after the go cue (1.3 s after delay onset):", bool((LT_FN.bin_start_from_delay >= 1.3).all()))
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(BIN_STARTS + 0.2, FRAC_FN, label="fastigial")
ax.plot(BIN_STARTS + 0.2, SIG_DN["all"].mean(0), label="dentate")
ax.axvspan(0, g.ACTIVATION_S, color="c", alpha=0.2); ax.axvline(1.3, color="k", ls="--", lw=0.5)
ax.set_xlabel("bin centre from delay onset (s)"); ax.set_ylabel("fraction p < 0.01"); ax.legend()
print("plotted range: fastigial", FRAC_FN.min().round(3), "to", FRAC_FN.max().round(3), "; dentate",
      SIG_DN["all"].mean(0).min().round(3), "to", SIG_DN["all"].mean(0).max().round(3))
subj_fn = np.concatenate([[p.subject] * len(p.spikes) for p in P if p.group == "fastigial"])
late_any = SIG_FN["all"][:, BIN_STARTS >= g.ACTIVATION_S + 1.0 - 1e-9]
print("per mouse, fraction significant in the first qualifying bin:\n",
      pd.Series(late_any[:, 0]).groupby(subj_fn).agg(["mean", "size"]).round(3))

**Result: reproduced.** In the bin starting 1.045 s after the end of fastigial photo-activation, 73 of 557 neurons (13.1%) differ from control, against 1% by chance (p = 3.7e-56) and 1.48% before onset; every bin up to the end of the response epoch stays above both (8.3% to 13.1%; correct trials 4.9% to 8.1%), and above the last pre-onset bin alone. All six mice contribute (5.0% to 18.7% in the first bin). Every qualifying bin starts after the go cue, so the test does not isolate preparatory activity; with lick right and lick left correct trials tested apart, so that the larger share of errors on photo-activation trials cannot drive it, 5.2% to 8.6% of neurons stay significant against 2.15% before onset.

### fig4g-dentate-effect-recovers (recovery)

Criterion: the fraction of significant neurons in the bins starting 1 s or more after offset is not above
the comparison of fig4f (same test, p >= 0.05) and is lower than the fastigial fraction in the same bins.

In [ ]:
for tr in ("all", "correct"):
    LTd, fpre_d, _ = late_tests(SIG_DN[tr])
    LTf, _, _ = late_tests(SIG_FN[tr])
    LTd["fastigial_fraction"] = LTf.fraction.values
    LTd["lower_than_fastigial"] = LTd.fraction < LTd.fastigial_fraction
    print(f"dentate, trials {tr}: neurons {len(SIG_DN[tr])}; pre-onset fraction {fpre_d:.4f}")
    print(LTd.to_string())
    print("first qualifying bin: not above chance", LTd.p_vs_chance.iloc[0] >= 0.05, "; not above pre-onset",
          LTd.p_vs_pre.iloc[0] >= 0.05, "; lower than fastigial", bool(LTd.lower_than_fastigial.iloc[0]),
          "| all bins not above (a):", bool((LTd.p_vs_chance >= 0.05).all()), "(b):", bool((LTd.p_vs_pre >= 0.05).all()),
          "; all lower than fastigial:", bool(LTd.lower_than_fastigial.all()))
    if tr == "all":
        LT_DN = LTd
LTt, fpre_t, _ = late_tests(SIG_DN_TYPE)
print(f"dentate, variant within trial type (correct trials): pre-onset fraction {fpre_t:.4f}; qualifying bins "
      f"{LTt.fraction.min():.4f} to {LTt.fraction.max():.4f}; bins above (a) {int((LTt.p_vs_chance < 0.05).sum())}, "
      f"(b) {int((LTt.p_vs_pre < 0.05).sum())}, last pre-onset bin {int((LTt.p_vs_last_pre_bin < 0.05).sum())} of {len(LTt)}")

**Result: not reproduced.** Dentate photo-activation modulates 165 of 525 neurons (0.314), inside the count tolerance but just below the fraction's (0.315 to 0.385). The latency by the paper's method is 0.0 to 1.4 ms, as the sign-flipped change is already 0.59 spikes/s above zero before onset; with that offset subtracted it is 3.4 to 10.4 ms, inside 8.3 +/- 5.0. After the photostimulus, 3.4% to 5.3% of neurons stay significant in every bin from 1 s after offset, above chance (1%) and the pre-onset 1.56% (p < 0.001), so ALM activity does not recover by the claim's test, although these fractions are lower than the fastigial ones in every bin (8.3% to 13.1%). With lick right and lick left correct trials tested apart (a variant, not a reading of the claims file), 2.1% to 4.8% stay significant, above the pre-onset 2.12% in 4 of 15 bins but above chance (1%) in 15 of 15: compared with the pre-onset level, much of the late dentate effect follows the mix of lick directions, while every bin stays above chance. Without the five sessions of the clock check the result is the same (3.5% to 5.4%, every bin above both comparisons).

### fig4e-fastigial-activation-abolishes-alm-selectivity

> Fastigial photo-activation abolishes selective persistent and ramping activity in most ALM neurons,
> while in a minority it creates new preparatory activity.

Criterion: population selectivity as E3 over the selective ALM neurons tested on > 3 trials in all
conditions (n = 328 in the paper), control against fastigial photo-activation trials, in the delay after
the photostimulus (readings (a) from photostimulus offset to the go cue, (b) the last 500 ms of the delay).
Per neuron (a choice named in the claims file), abolished means selectivity significant in control (E2
test on that window) and not on photo-activation trials; new means the reverse. Reproduced when under one
reading the population selectivity is lower on photo-activation trials (bootstrap 95% interval of the
difference excludes zero), more than half of the control-selective neurons lose significance, and the
neurons gaining it are fewer than those losing it. The per-neuron test has far fewer photo-activation
trials than control trials, so a loss of significance is partly a loss of power; a control with the same
number of control trials is shown.

In [ ]:
rng = np.random.default_rng(404)
rows = []
for per_type in (False, True):
    PS4, nsel = photo_selectivity("fastigial", ["2"], ["delay_after_act", "delay_last500"], rng, per_type)
    for w, d in PS4.items():
        diff = d.stim - d.ctl
        dm, dlo, dhi = ci_row(diff, rng)
        lose = (d.p_ctl < 0.05) & (d.p_stim >= 0.05)
        gain = (d.p_ctl >= 0.05) & (d.p_stim < 0.05)
        ctl_sel = int((d.p_ctl < 0.05).sum())
        rows.append(dict(pref="5 of each" if not per_type else "10 of each", window=w, selective=nsel, neurons=len(d),
                         control=round(d.ctl.mean(), 3), photo=round(d.stim.mean(), 3), diff=dm, diff_ci=(dlo, dhi),
                         control_selective=ctl_sel, lose=int(lose.sum()), lose_fraction=round(lose.sum() / max(ctl_sel, 1), 3),
                         gain=int(gain.sum()),
                         reproduced=(dhi < 0) and lose.sum() > ctl_sel / 2 and gain.sum() < lose.sum()))
    if not per_type:
        PS4_MAIN = PS4
R4E = pd.DataFrame(rows)
print(R4E.to_string())


# power control: the same per-neuron test on a random subset of control trials as large as the
# photo-activation set, against the rest of the control trials' significance
def power_control(group, codes, window, rng):
    lost = tot = 0
    for p in P:
        if p.group != group:
            continue
        sel, _ = g.selective(p, "correct", 1.3)
        t = p.t
        lick = t.lick.to_numpy()
        ctl = lick & (t.stim == "0").to_numpy()
        st = lick & t.stim.isin(codes).to_numpy()
        R = (t.instructed == "R").to_numpy(); L = ~R
        a, b = g.epoch_windows(t)[window]
        for j in np.where(sel)[0]:
            rate = p.ep[window][j] / (b - a)
            if g.ttest_p(rate[ctl & R], rate[ctl & L]) >= 0.05:
                continue
            nR, nL = (st & R).sum(), (st & L).sum()
            iR, iL = np.where(ctl & R)[0], np.where(ctl & L)[0]
            if nR < 4 or nL < 4 or len(iR) < nR or len(iL) < nL:
                continue
            sR, sL = rng.choice(iR, nR, replace=False), rng.choice(iL, nL, replace=False)
            tot += 1
            lost += g.ttest_p(rate[sR], rate[sL]) >= 0.05
    return lost, tot


for w in ("delay_after_act", "delay_last500"):
    lost, tot = power_control("fastigial", ["2"], w, rng)
    print(f"power control, {w}: control-selective neurons that lose significance on a control subset of the "
          f"photo-activation size: {lost} of {tot} = {lost / max(tot, 1):.3f}")

**Result: reproduced.** Over 208 selective neurons tested on more than 3 trials in all conditions (paper 328), fastigial photo-activation lowers population selectivity from 1.627 to 0.431 spikes/s from offset to the go cue (difference -1.196, interval -1.745 to -0.716), and 107 of 124 control-selective neurons lose significance against 6 that gain it; the last 500 ms and 10 preference trials per type give the same. Part of the loss is power: a control-trial subset as large as the photo-activation set loses significance in 93 of 124 (0.750) such neurons (78 of 120, 0.650, in the last 500 ms), against 107 of 124 (0.86) and 105 of 120 (0.88) with photo-activation, and the small photo-activation trial count also holds down the count that gains significance. So only part of the per-neuron loss is the effect itself, and the outcome rests mainly on the population difference and its interval.

## ALM population dynamics along the coding direction (Fig. 4h, 4i; Extended Data Fig. 10d)

E6: sessions with > 5 simultaneously recorded putative pyramidal neurons and > 5 trials per condition
(conditions: instructed side x control or early delay photo-activation, trials with a lick); rates in 400
ms windows in 10 ms steps, aligned to the go cue. The cd is the mean "lick right" minus "lick left"
response vector of correct control trials, averaged over the delay epoch, from a random half of the
correct control trials (the split is not given; a random half is the claims file's standard choice). The
cd is scaled to unit length, the convention of the method's source (Li et al. 2016, ref. 20), which
changes no sign or interval-excludes-zero test within a session. Projections use the other control
trials and the photo-activation trials, correct and error, grouped by instructed side; each session is
offset by the mean projection of its cd trials over all times. Means across sessions with bootstrap over
sessions. The paper counts 28 of 39 sessions.

In [ ]:
CD_T0, CD_T1 = -3.0, 1.3


def cd_session(p, codes, rng, window_rates=None):
    """Rates [neurons, trials, times] and the cd split for one session; None if it fails E6."""
    t = p.t
    lick = t.lick.to_numpy()
    R = (t.instructed == "R").to_numpy(); L = ~R
    ctl = lick & (t.stim == "0").to_numpy()
    st = lick & t.stim.isin(codes).to_numpy()
    if len(p.spikes) <= 5 or min((ctl & R).sum(), (ctl & L).sum(), (st & R).sum(), (st & L).sum()) <= 5:
        return None
    cor = t.correct.to_numpy()
    cR, cL = np.where(ctl & cor & R)[0], np.where(ctl & cor & L)[0]
    if min(len(cR), len(cL)) < 4:
        return None
    eR = rng.choice(cR, len(cR) // 2, replace=False); eL = rng.choice(cL, len(cL) // 2, replace=False)
    est = np.zeros(len(t), bool); est[eR] = True; est[eL] = True
    cue = t.cue_time.to_numpy()
    X = []
    for spk in p.spikes:
        c, C = g.binned(spk, cue, CD_T0, CD_T1, 0.4, 0.01)
        X.append(C / 0.4)
    X = np.array(X)   # neurons, trials, times
    delay = (c >= -1.3 + 0.2 - 1e-9) & (c <= -0.2 + 1e-9)   # windows inside the delay epoch
    cdt = X[:, est & R][:, :, delay].mean(1) - X[:, est & L][:, :, delay].mean(1)
    cd = cdt.mean(1)
    cd = cd / np.linalg.norm(cd)
    proj = np.einsum("n,nkt->kt", cd, X)
    offset = proj[est].mean()
    return dict(p=p, c=c, X=X, cd=cd, proj=proj - offset, est=est, ctl=ctl, st=st, R=R, L=L, cor=cor, delay=delay)


rng = np.random.default_rng(501)
CDS = {}
for p in P:
    if p.group in ("fastigial", "dentate"):
        r = cd_session(p, g.delay_code(p.group), rng)
        if r is not None:
            CDS[p.path] = r
CD_C = next(iter(CDS.values()))["c"]
print("sessions meeting E6:", len(CDS), "of", sum(p.group in ("fastigial", "dentate") for p in P),
      "; mice:", len({v["p"].subject for v in CDS.values()}),
      "; neurons per session", min(len(v["cd"]) for v in CDS.values()), "to", max(len(v["cd"]) for v in CDS.values()))
print(pd.Series({k.split("/")[1][:-4]: v["p"].group for k, v in CDS.items()}).value_counts())


def separation(r, which):
    """Right minus left mean projection, independent trials only, on control or stim trials."""
    m = (r["ctl"] & ~r["est"]) if which == "control" else r["st"]
    return r["proj"][m & r["R"]].mean(0) - r["proj"][m & r["L"]].mean(0)

### fig4i-cd-control-separation

> In control trials, ALM activity trajectories for "lick left" and "lick right" diverge along the coding
> direction during the sample epoch and stay separated through the delay epoch.

Criterion: separation (right minus left projection of control trials, independent of the cd), averaged
across sessions, bootstrap over sessions; reproduced when the 95% interval excludes zero at some time in
the sample epoch and at every 400 ms window centred within the delay epoch.

In [ ]:
rng = np.random.default_rng(502)
SEP_C = np.array([separation(r, "control") for r in CDS.values()])
m, lo, hi, _ = g.boot_mean_ci(SEP_C, rng, N_CI)
in_s = (CD_C > -2.6) & (CD_C < -1.3)
in_d = (CD_C > -1.3) & (CD_C < 0)
print("sample-epoch windows with the interval above 0:", int((lo[in_s] > 0).sum()), "of", int(in_s.sum()),
      "; first at", CD_C[in_s][lo[in_s] > 0][:1], "s from the go cue")
print("delay-epoch windows with the interval above 0:", int((lo[in_d] > 0).sum()), "of", int(in_d.sum()),
      "; smallest lower bound in the delay", round(lo[in_d].min(), 3))
print("separation (spikes/s along the unit cd) at -2.0, -1.0, -0.2 s:", np.round(np.interp([-2.0, -1.0, -0.2], CD_C, m), 3))
pre = CD_C < -2.8
print("before the sample epoch (centres < -2.8 s): mean", round(m[pre].mean(), 3), "; windows with interval above 0:",
      int((lo[pre] > 0).sum()), "of", int(pre.sum()))
by_m = pd.DataFrame(dict(subject=[r["p"].subject for r in CDS.values()], delay_sep=SEP_C[:, in_d].mean(1)))
print("per mouse, mean delay separation:\n", by_m.groupby("subject").delay_sep.agg(["mean", "size"]).round(2))
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.plot(CD_C, m); ax.fill_between(CD_C, lo, hi, alpha=0.3)
for x in (-2.6, -1.3, 0):
    ax.axvline(x, color="k", ls="--", lw=0.5)
ax.set_xlabel("time from go cue (s)"); ax.set_ylabel("R - L along cd")
print("plotted range", round(lo.min(), 2), "to", round(hi.max(), 2))

**Result: reproduced.** Over the 22 sessions meeting E6 (paper 28), the control separation along the cd has its 95% interval above zero from the first sample-epoch window (-2.59 s, 121 of 130 sample windows) and in all 130 delay windows (smallest lower bound 6.158), and not before the sample epoch (0 of 21).

### fig4i-fastigial-collapses-cd

> Fastigial photo-activation collapses ALM activity trajectories along the coding direction, while
> dentate photo-activation minimally affects them.

Criterion: separation in control, fastigial and dentate photo-activation trials, in the last 400 ms of the
delay (the window centred at -0.2 s) and averaged from photostimulus offset to the go cue (windows centred
from -0.645 to -0.2 s, inside offset to cue). "Collapse": (a) fastigial separation not distinguishable from
zero; (b) lower than control (interval of the difference over sessions excludes zero). Reproduced when
under one reading and one window the fastigial separation collapses and, in the same window, the dentate
separation does not differ from its control.

In [ ]:
rng = np.random.default_rng(503)
last = np.argmin(np.abs(CD_C - (-0.2)))
after = (CD_C >= -1.3 + g.ACTIVATION_S + 0.2 - 1e-9) & (CD_C <= -0.2 + 1e-9)
rows = []
for gr in ("fastigial", "dentate"):
    rs = [r for r in CDS.values() if r["p"].group == gr]
    sc = np.array([separation(r, "control") for r in rs])
    ss = np.array([separation(r, "stim") for r in rs])
    for wname, sel in (("last 400 ms", lambda x: x[:, last]), ("offset to cue", lambda x: x[:, after].mean(1))):
        a, b = sel(sc), sel(ss)
        ca, cb, cdf = ci_row(a, rng), ci_row(b, rng), ci_row(b - a, rng)
        rows.append(dict(group=gr, window=wname, sessions=len(rs), control=ca[0], control_ci=ca[1:], photo=cb[0],
                         photo_ci=cb[1:], diff=cdf[0], diff_ci=cdf[1:],
                         photo_includes_0=cb[1] <= 0 <= cb[2], diff_excludes_0=not (cdf[1] <= 0 <= cdf[2])))
R4I = pd.DataFrame(rows)
print(R4I.to_string())
for wname in ("last 400 ms", "offset to cue"):
    f_ = R4I[(R4I.group == "fastigial") & (R4I.window == wname)].iloc[0]
    d_ = R4I[(R4I.group == "dentate") & (R4I.window == wname)].iloc[0]
    print(f"{wname}: fastigial collapse (a) {f_.photo_includes_0}, (b) {f_.diff_excludes_0}; dentate unchanged {not d_.diff_excludes_0}")

**Result: reproduced** under reading (b). Fastigial photo-activation lowers the separation in the last 400 ms of the delay from 17.255 to 9.136 (difference -8.119, interval -15.118 to -1.551) and from offset to the go cue by 8.310 (-14.07 to -3.15), while dentate photo-activation leaves it unchanged (differences -0.683 and -1.704, intervals including 0). Under reading (a) the fastigial separation does not collapse to zero (interval 2.526 to 17.179).

### cd-captures-selectivity

> The projection along the coding direction captures 64.1% of ALM population selectivity over the sample
> and delay epochs and 27.0% of the total variance in task-related activity.

Criterion: per session, control trials over the sample and delay epochs. Selectivity: RMS of the
"lick right" minus "lick left" rate difference; variance: RMS of the baseline-subtracted activity
(baseline the 500 ms before sample onset, the claims file's standard choice). Readings: (a) the ratio of
the RMS along the unit cd to the RMS over all neurons; (b) the ratio of their squares. Mean and s.e.m.
across sessions; reproduced when under one reading both are within tolerance (64.1 +/- 3.4%, 27.0 +/-
4.2%). The trials are correct control trials, all of them (the cd is then estimated on the same trials)
and, as a variant, the independent half only. "Activity" is the trial-type mean trajectories (lick right
and lick left correct trials), a choice; single-trial activity is a variant.

In [ ]:
def captured(r, trial_set, single_trials=False):
    t_sd = (CD_C > -2.6 + 0.2 - 1e-9) & (CD_C < 0 - 0.2 + 1e-9)
    X, cd = r["X"], r["cd"]
    m = r["ctl"] & r["cor"]
    if trial_set == "independent":
        m = m & ~r["est"]
    xR, xL = X[:, m & r["R"]].mean(1), X[:, m & r["L"]].mean(1)
    d = (xR - xL)[:, t_sd]
    sel_all = np.sqrt((d ** 2).sum(0).mean()); sel_cd = np.sqrt(((cd @ d) ** 2).mean())
    # baseline: mean rate in the 500 ms before sample onset, per neuron, from spike counts
    p = r["p"]; t = p.t
    pin = t.pole_in_time.to_numpy()
    bl = np.array([g.counts(spk, pin[m] - 0.5, pin[m]).mean() / 0.5 for spk in p.spikes])
    if single_trials:
        A = X[:, m][:, :, t_sd] - bl[:, None, None]
        tot = np.sqrt((A ** 2).sum(0).mean()); along = np.sqrt((np.einsum("n,nkt->kt", cd, A) ** 2).mean())
    else:
        A = np.concatenate([xR[:, t_sd], xL[:, t_sd]], 1) - bl[:, None]
        tot = np.sqrt((A ** 2).sum(0).mean()); along = np.sqrt(((cd @ A) ** 2).mean())
    return sel_cd / sel_all, along / tot


rows = []
for trial_set in ("all", "independent"):
    for single in (False, True):
        v = np.array([captured(r, trial_set, single) for r in CDS.values()])
        for reading, f in (("a, RMS ratio", lambda x: x), ("b, squared", lambda x: x ** 2)):
            vv = 100 * f(v)
            rows.append(dict(trials=trial_set, activity="single trials" if single else "trial-type means", reading=reading,
                             sessions=len(vv), selectivity_pct=round(vv[:, 0].mean(), 1), sel_sem=round(stats.sem(vv[:, 0]), 1),
                             variance_pct=round(vv[:, 1].mean(), 1), var_sem=round(stats.sem(vv[:, 1]), 1),
                             within=bool(abs(vv[:, 0].mean() - 64.1) <= 3.4 and abs(vv[:, 1].mean() - 27.0) <= 4.2)))
R_CAP = pd.DataFrame(rows)
print(R_CAP.to_string())
print("reference: a random unit direction captures on average 1/sqrt(n) of the RMS; n per session",
      sorted(len(r["cd"]) for r in CDS.values()))

**Result: partially reproduced.** With the cd from one half of the correct control trials and the measures on the other half, as the paper does for every projection, reading (b) gives 62.1 +/- 4.7% of the selectivity, inside 64.1 +/- 3.4. The variance, with activity as the trial-type mean trajectories (the reading chosen here), is 34.2 +/- 4.7%, outside 27.0 +/- 4.2; only single-trial activity gives 24.0 +/- 2.9%, inside. The paper does not say which activity its total variance is over, so only the selectivity part holds under the chosen reading. With all trials (cd and measure on the same trials) the selectivity is 73.1% (b) or 84.9% (a).

### ed10d-fastigial-endpoints-no-prediction

> After fastigial photo-activation, the distance of ALM trajectories to the endpoints along the coding
> direction no longer predicts future movements.

Criterion: a claim of absence; the projection cd'x in the last 400 ms of the delay on fastigial
photo-activation trials (cd from independent correct control trials), the probability of licking right
against the projection, pooled over sessions; logistic regression of lick direction on the projection,
slope p < 0.05. Reproduced when the slope is not significant; the control-trial slope is shown. To pool
sessions, each projection is expressed as its position between the session's endpoints (0 at the mean
"lick left" and 1 at the mean "lick right" projection of the cd trials), the "distance to the endpoints"
of Extended Data Fig. 10c. The paper's n is 337 fastigial photo-activation trials.

In [ ]:
rows = []
for which in ("stim", "control"):
    xs, ys, ss = [], [], []
    for r in CDS.values():
        if r["p"].group != "fastigial":
            continue
        pr = r["proj"][:, last]
        eR = pr[r["est"] & r["R"]].mean(); eL = pr[r["est"] & r["L"]].mean()
        m = r["st"] if which == "stim" else (r["ctl"] & ~r["est"])
        choice = r["p"].t.choice.to_numpy()
        xs.append((pr[m] - eL) / (eR - eL)); ys.append((choice[m] == "R").astype(float)); ss.append([r["p"].path] * int(m.sum()))
    x, y = np.concatenate(xs), np.concatenate(ys)
    fit = sm.Logit(y, sm.add_constant(x)).fit(disp=0)
    ci = fit.conf_int()[1]
    bins = np.quantile(x, np.linspace(0, 1, 6))
    pb = [y[(x >= a) & (x <= b)].mean() for a, b in zip(bins[:-1], bins[1:])]
    rows.append(dict(trials=which, n=len(x), sessions=len(xs), slope=round(fit.params[1], 3), ci=(round(ci[0], 3), round(ci[1], 3)),
                     p=fit.pvalues[1], p_right_by_quintile=np.round(pb, 2), x_range=(round(x.min(), 2), round(x.max(), 2))))
R10D = pd.DataFrame(rows)
print(R10D.to_string())
rng = np.random.default_rng(504)
print("performance on these fastigial photo-activation trials against control (B1):",
      pd.DataFrame([perf_test(d2[d2.session.isin([k for k, r in CDS.items() if r["p"].group == "fastigial"])], rng,
                              "fastigial early delay, E6 sessions")]).to_string())

**Result: not reproduced.** On 215 fastigial photo-activation trials in 9 sessions (paper 337), the position between the endpoints still predicts the lick direction: logistic slope 0.420 (interval 0.128 to 0.713, p = 0.0049), weaker than on control trials (1.212, p = 1.1e-28). The probability of licking right rises from 0.30 to 0.58 across quintiles of the projection, against 0.25 to 0.84 in control.

## Sensitivity: the five sessions without a shown locked response

The clock check (data check) did not show a response locked to the trial events in five sessions
(BAYLORCD12 20180125 and 20180129, BAYLORCD13 20171211 and 20171215, anm00314756 20151023). They enter
the inhibition claims (CD12, CD13) and the dentate claims (anm00314756). Those claims are re-run without
them; the other claims do not use them.

In [ ]:
LOCK_FAIL = set(CK.loc[~CK.shown, "session"])
P_ALL = P
P = [p for p in P_ALL if p.path.split("/")[1][:-4] not in LOCK_FAIL]
rng = np.random.default_rng(601)
print("sessions left out:", sorted(LOCK_FAIL))
for name, manip in (("files", None), ("Supplemental Table 1", table_side)):
    bt = bias_table(behavior_trials(["inhibition"], ["5", "6"], manip=manip), rng, n_boot=2000)
    print(f"fig1k-inhibition-delay-bias, sides from {name}:\n",
          bt[["subject", "contra_change", "contra_p", "ipsi_change", "ipsi_p", "contra_bias_mouse", "ipsi_bias_mouse"]].to_string())
print("fig1g-dentate-activation-little-effect:", perf_test(behavior_trials(["dentate"], ["4"]), rng, "dentate", n_boot=2000))
mx = modulation("inhibition", ["5", "6"], "delay")
print("fig3d:", summarize_mod(mx, 454, 197), "\n latency:\n", latency(mx, rng, n_boot=1000)[0].to_string())
print(f"fig3e: all neurons control {mx.ctl_rate.mean():.3f}, photo-inhibition {mx.stim_rate.mean():.3f}, "
      f"unpaired p {stats.ttest_ind(mx.ctl_rate, mx.stim_rate).pvalue:.3f}, paired p {stats.ttest_rel(mx.ctl_rate, mx.stim_rate).pvalue:.3f}; "
      f"increase fraction {(mx[mx.modulated].change > 0).mean():.3f}")
rep = []
for seed in range(20):
    r_ = np.random.default_rng(6020 + seed)
    PSx, _ = photo_selectivity("inhibition", ["5", "6"], ["delay", "delay_last500"], r_, False)
    for w, d in PSx.items():
        c, s_ = ci_row(d.ctl, r_), ci_row(d.stim, r_)
        rep.append(dict(window=w, n=len(d), photo=s_[0], photo_lo=s_[1], reproduced=(c[1] > 0) and (s_[1] <= 0 <= s_[2])))
print("fig3f, 5 of each type, 20 draws:\n", pd.DataFrame(rep).groupby("window").agg(
    n=("n", "first"), reproduced=("reproduced", "sum"), photo_mean=("photo", "mean"), photo_lo_max=("photo_lo", "max")).round(3).to_string())
md_ = modulation("dentate", ["4"], "act_period")
print("fig4g:", summarize_mod(md_, 466, 163))
LTx, fprex, _ = late_tests(bin_fraction("dentate", ["4"]))
print(f"fig4g recovery: pre-onset {fprex:.4f}; qualifying bins {LTx.fraction.min():.4f} to {LTx.fraction.max():.4f}; "
      f"bins above chance {int((LTx.p_vs_chance < 0.05).sum())} and above pre-onset {int((LTx.p_vs_pre < 0.05).sum())} of {len(LTx)}")
P = P_ALL

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

| claim | outcome |
|---|---|
| fig1g-fastigial-activation-early-sample-no-effect | reproduced |
| fig1g-fastigial-activation-delay-errors | reproduced |
| fig1h-fastigial-activation-delay-bias | reproduced (reading a) |
| fig1g-dentate-activation-little-effect | reproduced (3 of 8 mice) |
| fig1k-inhibition-delay-bias | partially reproduced |
| fig2-alm-selective-fraction | reproduced |
| fig2cd-selectivity-emerges-gradually | partially reproduced (peak in the response epoch under one epoch only) |
| fig3d-alm-altered-by-cn-inhibition | not reproduced (count and latency) |
| fig3e-alm-mean-rate-unchanged | partially reproduced (assumed: unpaired test) |
| fig3e-alm-increase-decrease | partially reproduced (fraction, not counts) |
| fig3f-alm-selectivity-abolished-by-cn-inhibition | reproduced (most draws, last 500 ms) |
| fig4-fastigial-activation-alm-latency | partially reproduced (fewer modulated) |
| fig4e-fastigial-activation-abolishes-alm-selectivity | reproduced |
| fig4f-fastigial-effect-persists | reproduced |
| fig4g-dentate-effect-recovers | not reproduced |
| fig4i-cd-control-separation | reproduced |
| fig4i-fastigial-collapses-cd | reproduced |
| cd-captures-selectivity | partially reproduced (selectivity, not variance; assumed: independent trials, trial-type means) |
| ed10d-fastigial-endpoints-no-prediction | not reproduced |
| the other 26 claims (lesions, CN recordings, licking, sub-epoch and response epoch stimulation, VGAT-ChR2 experiments, anatomy) | blocked, data_missing |

**How the analysis differs from the paper's.** The photostimulation codes are not described in the files
and were read from the timing of the ALM response (data check). The behavior claims use only the
recording sessions, the only behavior in the dataset. Putative pyramidal neurons are classed by the
width of the files' mean waveforms, which gives more neurons than the paper in every group (1082 against
912 with CN photo-activation, 676 against 454 with photo-inhibition). The criteria of Figs. 3 and 4 judge a fraction only where the
dataset holds fewer neurons, so there the counts are judged and the fractions reported; the units table has no quality
field that would tell which neurons the paper left out. Selectivity tests use control trials; lick early trials are left out of every neural analysis, and so
are 120 trials (without an early lick) of the last 127 of BAYLORCD8 20171020, recorded after its units' observation intervals end.
The latency's bin label, the preference trials of E3, the cd's normalisation and the trials for the
captured fraction are not in the paper; every reading run is shown above. Leaving out the five sessions
whose locked response the clock check could not show changes no outcome (section Sensitivity).

**What was found about the dataset.** It holds the ALM recordings of the 13 ALM-recorded mice of
Supplemental Table 1 (54 sessions, against 39 + 19 = 58 in the paper), with no CN recordings, no lick
times, no behavior-only sessions and no anatomy. `photo_stim_id` has no description. The event-time
columns are described as relative to the trial start but are on the session clock. For CD12 and CD13 the
files place the manipulated CN on the same side as the recorded ALM (left), and for CD14 the recorded ALM
on the left, where Supplemental Table 1 gives the right CN for CD12 and CD13, the right ALM for CD14, and
says ALM was always recorded contralaterally. The units table has no quality field, so nothing in the files explains why they hold more putative
pyramidal neurons than the paper; in one session (BAYLORCD8 20171020) the behavior runs on past the end
of the units' observation intervals, and those trials hold no spikes. The files' `cell_type` labels 60 units "pyramidal" whose
waveform width is under 0.45 ms. Subject names (anm00314746 to anm00314760) do not match the paper's
(AN105 to AN109), and genotype is "unknown" for the wild-type mice.

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")